# KíLín — Literature-Project Evaluation Notebook

**Routing, conversational scope, retrieval, scientific evidence, and model
comparisons**

This notebook creates an independent literature workspace from five PDFs in
`tests/fixtures`, exercises KíLín through its normal `RAGSession` API, and
records expected and observed behavior. Its purpose is to distinguish faults
in routing, context selection, evidence retrieval, and answer generation.
It is **not** an automated scientific-correctness judge.

### Evaluation dimensions

1. **Routing:** correct intent, configured tool, retrieval policy, and
   follow-up links. A routing-only probe does not execute a tool.
2. **Document scope:** expected PDFs occur in retained evidence; a
   tool may scan additional files without retaining passages from them.
   Scanned files and evidence sources are reported separately.
3. **Execution and citations:** token limits, incomplete outputs, and
   citation-identifier warnings are recorded independently of routing.
   Quoted bibliography numbers can produce false-positive citation warnings;
   inspect the actual source text before adjudicating them.
4. **Scientific content:** answer keywords and source coverage flag
   potential errors; reference facts identify claims for manual review.
   Neither check independently establishes scientific correctness.
5. **Model dependence:** controlled comparisons vary answering or routing
   models while retaining the same documents and questions.

### Recorded answer summaries

KíLín creates a compact representation of each completed turn for routing.
The notebook **reads its summary directly from the native
`conversation.turn.representation` trace event** (or from the corresponding
saved turn checkpoint if the event is unavailable). It never independently
summarizes the raw answer. The compact table shows that recorded summary;
**Answer and routing details** retains the entire unmodified response,
including model reasoning. Trace/checkpoint differences are flagged. Generation-stage model events
and tool-reported internal attempts are counted separately; neither alone
represents every call made by a backend.

`---` means that no expected intent is assigned, no tool was invoked, or a
particular check is not applicable. It does **not** indicate success.
`MATCH` and `REVIEW` evaluate declared routing expectations only; separate
scope, execution, trace, and content flags are shown alongside them.

### How to run

Restart the kernel and execute **one section at a time, in order**. This
ensures the saved outputs correspond to the current reporting helpers. Each
case prints a small table, with the full response and trace diagnostics
collapsed beneath it. Use `inspect_case("CASE_ID")` for detailed inspection.
Tool-heavy tests, additional source checks, and model comparisons are opt-in.

Traces may contain private source text and model reasoning. The notebook
stores results locally, cancels network operations, and does not modify KíLín
source code or project-type configuration.

## 1. Experiment configuration

Choose the answering model and router independently. The default settings
keep time-consuming tools and model comparisons disabled; enable them one
section at a time when needed. The workspace name includes a timestamp to
avoid overwriting evidence from another experiment.

The model must be available locally when `ALLOW_DOWNLOAD=False`.
Keep retrieval count and generation settings fixed when comparing models;
change one experimental variable at a time.

In [ ]:
from datetime import datetime
from pathlib import Path

# The run directory is new; set a fixed ID to make a repeat reproducible.
EXPERIMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PROJECT_NAME = f"trial_literature_routing_{EXPERIMENT_ID}"
ALLOW_DOWNLOAD = False

MODEL_IDS = {
    "qwen8b": "Qwen/Qwen3-8B-MLX-4bit",
    "qwen14b": "Qwen/Qwen3-14B-MLX-4bit",
    "qwen38_27b": "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly",
}
PRIMARY_MODEL = "qwen8b"
ROUTER_MODEL = "Qwen/Qwen3-4B-MLX-4bit"
COMPARISON_MODELS = ("qwen8b", "qwen38_27b")
COMPARISON_ROUTERS = (
    "Qwen/Qwen3-4B-MLX-4bit",
    "Qwen/Qwen3-8B-MLX-4bit",
)

RETRIEVAL_COUNT = 10
RUN_LONG_TOOLS = False
RUN_SOURCE_CHECKS = False
RUN_REPORTS = False
RUN_MODEL_COMPARISON = False
RUN_ROUTER_COMPARISON = False
RUN_GUI_EXPORT_TEST = False

# Inline answers stay compact; expanded details show complete responses.

assert PRIMARY_MODEL in MODEL_IDS
assert all(name in MODEL_IDS for name in COMPARISON_MODELS)
assert EXPERIMENT_ID.strip()

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook inside the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / PROJECT_NAME
print("Repository:", repo_root)
print("Experiment:", PROJECT_NAME)
print("Project path:", project_dir)
print("Answer model:", MODEL_IDS[PRIMARY_MODEL])
print("Router model:", ROUTER_MODEL)

Repository: /Users/giovannivolpe/Documents/GitHub/KiLin
Experiment: trial_literature_routing_20261102-203250
Project path: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-203250
Answer model: Qwen/Qwen3-8B-MLX-4bit
Router model: Qwen/Qwen3-4B-MLX-4bit


## 2. Evaluation and inspection helpers

The helpers save questions, selected intents, source evidence, token usage,
GUI choices, and native trace information without changing KíLín's behavior.
Routing-only probes call the router but do not execute tools or save answers.

The **recorded summary** comes directly from the KíLín turn-representation
trace event. Prior-answer snapshots use these same saved representations,
including their source labels; if a trace is missing, the corresponding
saved checkpoint is used and its origin reported. Neither summary is rebuilt
from the raw response. Expanded details retain the full untruncated answer.

Routing `MATCH` is independent of scope, content anchors, citations, and
model completion. File scans are distinct from retained evidence, and
generation-stage events from tool-internal model attempts. Citation
warnings are diagnostic, not automatically proof
of invented citations, because source quotations may contain bibliography
numbers. `---` means no expected route or no tool/check, not an intent named
`observe`.

In [2]:
import csv
import html
import json
from contextlib import contextmanager
from time import perf_counter

from IPython.display import HTML, display

from kilin.answering import route_question, strip_thinking
from kilin.conversations import ConversationStore
from kilin.project_type import project_type
from kilin.rag import RAGSession

CASE_RESULTS = []
CASE_INDEX = {}
CONVERSATION_INDEX = {}


def literature_intents():
    """Return configured literature intents without hard-coded tool names.

    Returns
    -------
    dict
        Current name-to-configuration mapping from literature.json.

    """
    path = repo_root / "kilin" / "project_types" / "literature.json"
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)["intents"]


def expected_tool(intent):
    """Return the tool configured for an intent, or None for model answers.

    Parameters
    ----------
    intent : str or None
        Intent identifier in the current literature project type.

    Returns
    -------
    str or None
        Configured tool name, if present.

    """
    return literature_intents().get(intent, {}).get("tool", {}).get("name")


def _source_names(evidence):
    """Identify filenames in saved source evidence, retaining order.

    Parameters
    ----------
    evidence : sequence
        Evidence objects persisted by RAGSession.

    Returns
    -------
    list of str
        Unique document basenames.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if path:
            name = Path(str(path)).name
            if name not in found:
                found.append(name)
    return found


def _trace_operation(stages, stage_name, suffix):
    """Locate an operation by semantic suffix, not sequence number.

    Parameters
    ----------
    stages : dict
        One native trace turn's stages.
    stage_name : str
        Stage name such as ``01_question``.
    suffix : str
        End of the operation name, without its sequence prefix.

    Returns
    -------
    dict
        Matching operation or an empty mapping.

    """
    operations = stages.get(stage_name, {}).get("operations", {})
    return next(
        (value for key, value in operations.items() if key.endswith(suffix)),
        {},
    )


def _trace_turns(path):
    """Load committed turns from the native conversation trace.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace JSON path.

    Returns
    -------
    list of dict
        Recorded turns, or an empty list if unavailable.

    """
    try:
        with path.open(encoding="utf-8") as stream:
            trace = json.load(stream)
        turns = trace["sessions"]["05_conversations"]["turns"]
        return list(turns.values())
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return []


def _recorded_representation(turn, traced):
    """Read the KíLín-created summary, preferring its trace event.

    Parameters
    ----------
    turn : ConversationTurn
        Committed turn containing KíLín's saved representation.
    traced : dict or None
        Representation from ``conversation.turn.representation``.

    Returns
    -------
    dict
        Recorded summary, provenance, and trace/checkpoint agreement.

    Notes
    -----
    Neither source is reconstructed from the raw answer. If both recorded
    representations exist, they should agree on their semantic fields.

    """
    traced = traced if isinstance(traced, dict) else {}
    parameters = getattr(turn, "parameters", {}) or {}
    checkpoint = parameters.get("representation") or {}
    if not isinstance(checkpoint, dict):
        checkpoint = {}
    trace_ok = isinstance(traced.get("summary"), str)
    saved_ok = isinstance(checkpoint.get("summary"), str)
    if trace_ok:
        representation = traced
        origin = "trace"
    elif saved_ok:
        representation = checkpoint
        origin = "checkpoint"
    else:
        representation = {}
        origin = "unavailable"
    consistent = None
    if trace_ok and saved_ok:
        fields = ("question", "summary", "sources")
        consistent = all(
            traced.get(field) == checkpoint.get(field)
            for field in fields
        )
    return {
        "summary": representation.get("summary", ""),
        "sources": list(representation.get("sources") or []),
        "origin": origin,
        "consistent": consistent,
    }


def _native_route(path, turn_number):
    """Read routing and timing details from one saved native trace turn.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace path.
    turn_number : int
        One-based turn number in this conversation.

    Returns
    -------
    dict
        Router, history, retrieval, and model-call diagnostics.

    """
    turns = _trace_turns(path)
    if not 1 <= turn_number <= len(turns):
        return {"trace_available": False}
    stages = turns[turn_number - 1].get("stages", {})

    routed = _trace_operation(
        stages, "01_question", "intent_classification_decided"
    ).get("details", {})
    history = _trace_operation(
        stages, "01_question", "history_selection_result"
    ).get("details", {})
    retrieval = _trace_operation(
        stages, "03_retrieval", "retrieval_result"
    ).get("details", {})
    settings = _trace_operation(
        stages, "01_question", "turn_settings_snapshot"
    ).get("details", {})
    representation = _trace_operation(
        stages, "01_question", "conversation_turn_representation"
    ).get("details", {})
    model_calls = []
    for stage in stages.values():
        for name, value in stage.get("operations", {}).items():
            if name.endswith("llm_call_output"):
                model_calls.append(value.get("details", {}))
    return {
        "trace_available": True,
        "classification": routed.get("classification", {}),
        "shortcut": routed.get("shortcut"),
        "fallback": routed.get("fallback"),
        "dropped_turns": routed.get("dropped_turns", []),
        "router_raw": routed.get("raw"),
        "history": history,
        "retrieval": retrieval,
        "settings": settings,
        "model_calls": model_calls,
        "representation": representation,
    }

/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _gui_callbacks(events, choice="default", confirmation="quick"):
    """Create non-networked callbacks and record each scripted decision.

    Parameters
    ----------
    events : list
        Mutable list populated with GUI-equivalent interactions.
    choice : str or tuple, optional
        ``default``, ``first20``, ``save_all``, ``cancel``, or a tuple
        ``("document", filename)``. A value of ``none`` omits the adapter.
    confirmation : str, optional
        ``quick``, ``continue``, or ``cancel`` for model plans. Network plans
        are always cancelled.

    Returns
    -------
    tuple
        Choice callback and confirmation callback for ``RAGSession.ask``.

    """

    def choose(question, options):
        selected = None
        if isinstance(choice, tuple) and choice[0] == "document":
            target = choice[1].casefold()
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.split(" (", 1)[0].casefold() == target
                ),
                None,
            )
        elif choice in ("default", "first20"):
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item == "Show the first 20 here"
                ),
                None,
            )
            if selected is None and choice == "default" and options:
                if "which paper" in question.casefold() or (
                    "which document" in question.casefold()
                ):
                    selected = 0
        elif choice == "save_all":
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.startswith("Save all ")
                    and item.endswith(" to outputs/ as Markdown")
                ),
                None,
            )
        events.append(
            {
                "kind": "choice",
                "question": question,
                "options": list(options),
                "selected": selected,
            }
        )
        return selected

    def confirm(plan):
        decision = (
            "cancel"
            if getattr(plan, "operation", "model") == "network"
            else confirmation
        )
        events.append(
            {
                "kind": "confirmation",
                "tool": plan.tool,
                "operation": getattr(plan, "operation", "model"),
                "summary": plan.summary,
                "items": plan.items,
                "batches": plan.batches_to_run,
                "quick_items": plan.quick_items,
                "selected": decision,
            }
        )
        return decision

    choice_callback = None if choice == "none" else choose
    return choice_callback, confirm


@contextmanager
def conversation(label, *, model=None, router=None, branch_from=None):
    """Open an independent test conversation and close it reliably.

    Parameters
    ----------
    label : str
        Human-readable scenario group name.
    model : str or None, optional
        Key in MODEL_IDS; defaults to PRIMARY_MODEL.
    router : str or None, optional
        Router model identifier; defaults to ROUTER_MODEL.
    branch_from : str or None, optional
        Existing conversation ID whose saved history is cloned into this
        conversation. A new independent conversation is the default.

    Yields
    ------
    dict
        Active session, trace path, and case-to-turn registry.

    """
    selected = model or PRIMARY_MODEL
    if selected not in MODEL_IDS:
        raise ValueError(f"Unknown model key: {selected}")
    if label in CONVERSATION_INDEX:
        raise ValueError(f"Conversation label already used: {label}")

    store = ConversationStore(project_dir)
    if branch_from is not None:
        store.ensure()
    info = store.create(branch_from=branch_from)
    path = store.state_path(info)
    session = None
    try:
        session = RAGSession(
            project_dir,
            conversation_path=path,
            router_model=router or ROUTER_MODEL,
        )
        session.use_chat_model(MODEL_IDS[selected])
        context = {
            "label": label,
            "store": store,
            "info": info,
            "path": path,
            "session": session,
            "case_turn_ids": {},
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        CONVERSATION_INDEX[label] = {
            "path": str(path.relative_to(project_dir)),
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        print(f"Conversation {label}: {path.name}")
        print("  Answering:", session.chat_session.model_id)
        print("  Router:", session.router_model)
        yield context
    finally:
        if session is not None:
            session.close()


def _expected_links(context, case_ids):
    """Resolve named earlier cases to one-based conversation turn IDs.

    Parameters
    ----------
    context : dict
        Active conversation context.
    case_ids : tuple
        Previously run case identifiers.

    Returns
    -------
    list of int
        One-based expected turn numbers.

    """
    missing = [
        case_id
        for case_id in case_ids
        if case_id not in context["case_turn_ids"]
    ]
    if missing:
        raise ValueError(f"Missing earlier cases: {missing}")
    return [context["case_turn_ids"][item] for item in case_ids]


def _check_record(record):
    """Evaluate routing, source scope, and diagnostics independently.

    Parameters
    ----------
    record : dict
        Completed routing probe or committed conversation turn.

    Returns
    -------
    dict
        Separate lists of routing, scope, execution, trace, and content
        concerns. Content anchors are not scientific correctness proofs.

    """
    checks = {
        "routing": [],
        "scope": [],
        "execution": [],
        "trace": [],
        "content": [],
    }
    route = checks["routing"]
    expected = record.get("expected_intent")
    if expected is not None and record.get("intent") != expected:
        route.append("intent")
    if record["mode"] == "full" and expected is not None:
        if record.get("tool") != record.get("expected_tool"):
            route.append("tool")
    standalone = record.get("expected_standalone")
    if standalone is not None and record.get("standalone") != standalone:
        route.append("standalone")
    actual = set(record.get("related_turn_ids", []))
    required = set(record.get("expected_related_ids", []))
    forbidden = set(record.get("forbidden_related_ids", []))
    if not required.issubset(actual):
        route.append("missing prior-turn link")
    if forbidden & actual:
        route.append("unrelated prior-turn link")
    if record.get("expected_retrieval") is not None:
        if record.get("retrieval_mode") != record["expected_retrieval"]:
            route.append("retrieval policy")
    if record.get("intent") in record.get("disallowed_intents", []):
        route.append("disallowed intent")

    if record["mode"] == "full":
        sources = set(record.get("sources", []))
        if not set(record.get("expected_sources", [])).issubset(sources):
            checks["scope"].append("missing source in retrieved evidence")
        if set(record.get("forbidden_sources", [])) & sources:
            checks["scope"].append("out-of-scope retrieved source")
        if record.get("invalid_citations"):
            checks["execution"].append("citation warning (check quotes)")
        if record.get("completion"):
            checks["execution"].append("incomplete answer")
        if not record.get("trace", {}).get("trace_available"):
            checks["trace"].append("native trace unavailable")
        if record.get("summary_source") != "trace":
            checks["trace"].append("summary not from trace event")
        if record.get("summary_consistent") is False:
            checks["trace"].append("trace/checkpoint summary mismatch")
        logged_model = record.get("trace", {}).get(
            "settings", {}
        ).get("chat_model")
        if logged_model and logged_model != record.get("model"):
            checks["trace"].append("active-model trace mismatch")
        for prior in record.get("previous", []):
            if prior.get("summary_consistent") is False:
                checks["trace"].append("prior summary mismatch")
                break
        answer = strip_thinking(record.get("answer") or "").casefold()
        for term in record.get("expected_answer_terms", []):
            if term.casefold() not in answer:
                checks["content"].append(f"missing answer anchor: {term}")
    return checks


def _grade_record(record):
    """Attach independent diagnostic fields and a routing-only verdict.

    Parameters
    ----------
    record : dict
        Populated case record, updated in place.

    """
    checks = _check_record(record)
    record["checks"] = checks
    record["issues"] = [
        f"{kind}: {message}"
        for kind, problems in checks.items()
        for message in problems
    ]
    record["status"] = (
        "REVIEW"
        if checks["routing"]
        else "MATCH" if record.get("expected_intent") else "---"
    )
    record["scope_status"] = (
        "REVIEW" if checks["scope"]
        else "OK" if record.get("expected_sources")
        or record.get("forbidden_sources") else "---"
    )


In [4]:
def _brief(value, limit=95):
    """Return short escaped text for a compact diagnostic table.

    Parameters
    ----------
    value : object
        Value to show.
    limit : int, optional
        Maximum characters before truncation.

    Returns
    -------
    str
        Escaped, whitespace-normalized label.

    """
    text = " ".join(str(value if value is not None else "-").split())
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return html.escape(text)


def _final_answer_text(answer):
    """Remove reasoning for content checks, not answer summaries.

    Parameters
    ----------
    answer : str or None
        Raw answer, potentially containing model reasoning.

    Returns
    -------
    str
        Answer text excluding thinking blocks.

    """
    return strip_thinking(answer or "")


def show_case(record):
    """Show one compact table and optional full answer diagnostics.

    Parameters
    ----------
    record : dict
        Recorded routing-only or end-to-end test case.

    """
    expected = record.get("expected_intent") or "---"
    actual = record.get("intent") or "-"
    tool = record.get("tool") or "---"
    if record["mode"] == "route-only":
        tool = "---"
    else:
        wanted = record.get("expected_tool") or "---"
        tool = f"{wanted} → {tool}"
    related = record.get("related_turn_ids", [])
    need = record.get("expected_related_ids", [])
    tokens = record.get("usage", {})
    token_text = (
        f"{tokens.get('input_tokens', 0)}/" f"{tokens.get('output_tokens', 0)}"
        if tokens
        else "-"
    )
    source_names = record.get("sources", [])
    source_text = f"{len(source_names)}: " + ", ".join(source_names)
    scanned = record.get("files_scanned")
    if scanned is not None:
        source_text += f" ({scanned} scanned)"
    status = record["status"]
    checks = record.get("checks", {})
    other = ", ".join(
        name for name in ("scope", "execution", "trace", "content")
        if checks.get(name)
    ) or "---"
    body = "".join(
        "<td style='padding:4px 7px;border-bottom:1px solid #bbb'>"
        + value
        + "</td>"
        for value in (
            _brief(f"{expected} → {actual}", 55),
            _brief(tool, 22),
            _brief(f"need {need}; got {related}", 29),
            _brief(record.get("retrieval_mode") or "---", 12),
            _brief(source_text, 47),
            _brief(token_text, 22),
            _brief(f"{record.get('seconds', 0):.2f}s"),
            _brief(status, 18),
            _brief(other, 36),
        )
    )
    title = (
        f"<strong>{_brief(record['id'])}</strong> · "
        f"{_brief(record['question'], 145)}"
    )
    header = "".join(
        f"<th style='padding:4px 7px;text-align:left'>"
        f"{html.escape(name)}</th>"
        for name in (
            "Expected → actual",
            "Tool",
            "Prior links",
            "Retrieval",
            "Evidence",
            "Tokens in/out",
            "Time",
            "Routing check",
            "Other checks",
        )
    )
    answer = record.get("answer") or ""
    summary = record.get("answer_summary") or ""
    if record["mode"] == "route-only":
        summary = "---"
    elif not summary and record.get("summary_source") == "unavailable":
        summary = "(summary unavailable)"
    elif not summary:
        summary = "(no final-answer summary)"
    one_line_answer = _brief(summary, 180)
    issues = "<div>" + "</div><div>".join(
        f"<b>{html.escape(kind)}:</b> "
        f"{_brief(', '.join(messages), 300)}"
        for kind, messages in checks.items() if messages
    ) + "</div>" if any(checks.values()) else "none"
    origin = f"{record.get('label', '?')} / {record.get('model', '?')}"
    trace_path = record.get("trace_path", "-")
    route = record.get("trace", {})
    model_calls = route.get("model_calls", [])
    generation_events = record.get("generation_events")
    tool_attempts = record.get("tool_internal_attempts")
    tool_text = (
        str(tool_attempts) if tool_attempts is not None
        else "not reported" if record.get("tool") else "---"
    )
    limit_hits = sum(
        call.get("finish_reason") == "length" for call in model_calls
    )
    prior_turns = record.get("previous", [])
    chosen = set(record.get("related_turn_ids", []))
    prior_html = "".join(
        f"<li>{'*' if item['turn'] in chosen else ' '} "
        f"#{item['turn']} {_brief(item['question'], 100)}: "
        f"{_brief(item['summary'], 120)} "
        f"({_brief(item.get('summary_source'), 15)})</li>"
        for item in prior_turns
    )
    refers_back = route.get("classification", {}).get(
        "refers_back", route.get("refers_back")
    )
    scope_note = ""
    if scanned is not None:
        scope_note = (
            f"<div><b>Scan:</b> {scanned} files scanned; "
            f"{record.get('files_with_mentions')} with matches; "
            f"{record.get('listed')} items listed; "
            f"{len(source_names)} source files in saved evidence.</div>"
        )
    elif record.get("passages_read") is not None:
        scope_note = (
            "<div><b>Passages actually read:</b> "
            f"{record['passages_read']} "
            f"(planned scope: {record.get('planned_passages')})."
            "</div>"
        )
    details = (
        f"<div><b>Scenario:</b> {_brief(origin)} · "
        f"<b>Turn:</b> {_brief(record.get('turn_number'))} · "
        f"<b>Trace:</b> {_brief(trace_path, 130)}</div>"
        f"<div><b>Checks:</b> {issues}</div>"
        f"<div><b>Summary provenance:</b> "
        f"{_brief(record.get('summary_source'))} · "
        f"<b>Trace/checkpoint agree:</b> "
        f"{_brief(record.get('summary_consistent'))}</div>"
        f"<div><b>Configured tool:</b> "
        f"{_brief(expected_tool(record.get('intent')) or '---')} · "
        f"<b>Evidence sources:</b> {_brief(source_names, 350)}</div>"
        f"{scope_note}"
        f"<div><b>Shortcut:</b> {_brief(route.get('shortcut'))} · "
        f"<b>Fallback:</b> {_brief(route.get('fallback'))}</div>"
        f"<div><b>History IDs:</b> "
        f"{_brief(record.get('history_turn_ids', []))} · "
        f"<b>Refers back:</b> {_brief(refers_back)}</div>"
        f"<div><b>Generation-stage events:</b> "
        f"{generation_events if generation_events is not None else '---'}"
        f" · <b>Tool-internal attempts:</b> {_brief(tool_text)} · "
        f"<b>Length stops (generation stage):</b> {limit_hits} · "
        f"<b>Finish:</b> {_brief(record.get('finish_reason'))}</div>"
        "<details><summary>Earlier answer snapshots "
        "(* means linked by router)</summary><ul>"
        f"{prior_html}</ul></details>"
        "<div><b>Complete saved response:</b></div>"
        "<pre style='white-space:pre-wrap;max-height:300px;"
        "overflow:auto;font-size:12px'>"
        f"{html.escape(answer)}</pre>"
    )
    display(
        HTML(
            f"<div style='margin:8px 0 15px 0;font-size:12px'>"
            f"<div style='margin-bottom:5px'>{title}</div>"
            "<div style='overflow-x:auto'>"
            "<table style='border-collapse:collapse;width:100%;"
            "font-size:11px'><thead><tr>"
            f"{header}</tr></thead><tbody><tr>{body}</tr></tbody></table>"
            "</div>"
            f"<div style='margin:3px 0'><b>Recorded summary:</b> "
            f"{one_line_answer}</div>"
            f"<details style='margin-top:5px'><summary>Answer and routing "
            f"details</summary>{details}</details></div>"
        )
    )


def inspect_case(case_id, *, show_answer=True, show_prior=True):
    """Print the complete saved answer, related context, and trace data.

    Parameters
    ----------
    case_id : str
        ID shown by the compact diagnostic table.
    show_answer : bool, optional
        Whether to print the full answer, including any reasoning tags.
    show_prior : bool, optional
        Whether to print earlier linked questions and saved answer summaries.

    Returns
    -------
    dict
        Unmodified record collected by the evaluation harness.

    """
    record = CASE_INDEX[case_id]
    print(
        json.dumps(
            {
                key: value
                for key, value in record.items()
                if key not in ("answer", "trace")
            },
            indent=2,
            ensure_ascii=False,
        )
    )
    if show_prior:
        print("\nPREVIOUS TURN SNAPSHOT")
        for item in record.get("previous", []):
            marker = (
                "*"
                if item["turn"] in record.get("related_turn_ids", [])
                else " "
            )
            print(
                f"{marker} #{item['turn']}: {item['question']}\n"
                f"    {item['summary']}"
            )
    if show_answer:
        print("\nFULL ANSWER\n", record.get("answer", ""))
    print("\nNATIVE ROUTING DIAGNOSTICS")
    print(json.dumps(record.get("trace", {}), indent=2, ensure_ascii=False))
    return record

In [5]:
def _previous_snapshot(context):
    """Capture KíLín's recorded prior-turn summaries for the router.

    Parameters
    ----------
    context : dict
        Active conversation state.

    Returns
    -------
    list of dict
        Prior-turn metadata with authoritative summary provenance.

    Notes
    -----
    Read ``conversation.turn.representation`` events first. When a trace
    is unavailable, use the same representation saved in turn parameters.
    Do not independently summarize the raw assistant response.

    """
    snapshot = []
    native_turns = _trace_turns(context["path"])
    for index, turn in enumerate(context["session"].turns, start=1):
        stages = (
            native_turns[index - 1].get("stages", {})
            if index <= len(native_turns) else {}
        )
        traced = _trace_operation(
            stages, "01_question", "conversation_turn_representation"
        ).get("details", {})
        representation = _recorded_representation(turn, traced)
        snapshot.append(
            {
                "turn": index,
                "question": turn.question,
                "summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "sources": _source_names(turn.evidence),
                "intent": turn.parameters.get("intent"),
                "tool": (turn.parameters.get("tool") or {}).get("name"),
            }
        )
    return snapshot


def _register(record):
    """Retain and display one result without changing any test outcome.

    Parameters
    ----------
    record : dict
        New unique case record.

    """
    if record["id"] in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {record['id']}")
    CASE_RESULTS.append(record)
    CASE_INDEX[record["id"]] = record
    show_case(record)


def run_case(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    retrieval=None,
    sources=(),
    forbid_sources=(),
    disallow=(),
    answer_terms=(),
    choice="default",
    confirm="quick",
):
    """Run and record a full RAG turn with explicit context expectations.

    Parameters
    ----------
    context : dict
        Test conversation returned by ``conversation``.
    case_id : str
        Unique identifier used for later comparisons and follow-ups.
    question : str
        Unmodified user question.
    intent : str or None, optional
        Expected literature intent; None records an exploratory observation.
    standalone : bool or None, optional
        Expected standalone classification; None omits this check.
    related_to : tuple of str, optional
        Earlier case IDs whose turns must be linked by the router.
    forbid_related : tuple of str, optional
        Earlier case IDs the router should not link to this question.
    retrieval : str or None, optional
        Expected retrieval mode: search, reuse, document, or none.
    sources : tuple of str, optional
        Source filenames that must appear in saved evidence.
    forbid_sources : tuple of str, optional
        Source filenames that must not occur in saved evidence.
    disallow : tuple of str, optional
        Intents that must not be chosen, including ambiguous cases.
    answer_terms : tuple of str, optional
        Required final-answer keywords. A lightweight sanity check, not
        a semantic evaluation of the scientific interpretation.
    choice : str or tuple, optional
        Scripted GUI document/list selection.
    confirm : str, optional
        GUI-equivalent confirmation mode, with network access disabled.

    Returns
    -------
    dict
        Recorded answer and machine-verifiable routing observations.

    Notes
    -----
    A routing mismatch is recorded without stopping the conversation.
    Later turns therefore test how the system handles imperfect history.
    Missing required previous turns is a test setup error and raises.

    """
    if intent is not None and intent not in literature_intents():
        raise ValueError(f"Unknown intent: {intent}")
    expected_links = _expected_links(context, related_to)
    forbidden_links = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    before = len(session.turns)
    previous = _previous_snapshot(context)
    events = []
    choose, confirm_plan = _gui_callbacks(events, choice, confirm)
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "full",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": expected_links,
        "forbidden_related_ids": forbidden_links,
        "expected_retrieval": retrieval,
        "expected_sources": list(sources),
        "forbidden_sources": list(forbid_sources),
        "disallowed_intents": list(disallow),
        "expected_answer_terms": list(answer_terms),
        "previous": previous,
        "trace_path": str(context["path"].relative_to(project_dir)),
        "gui_events": events,
    }
    try:
        answer = session.ask(
            question,
            retrieval_count=RETRIEVAL_COUNT,
            choice_callback=choose,
            confirm_callback=confirm_plan,
        )
        if len(session.turns) != before + 1:
            raise RuntimeError("Expected exactly one committed turn.")
        turn = session.turns[-1]
        parameters = turn.parameters
        tool = parameters.get("tool") or {}
        native = _native_route(context["path"], len(session.turns))
        representation = _recorded_representation(
            turn, native.get("representation")
        )
        internal = tool.get("internal_generation") or {}
        attempts = internal.get("calls")
        record.update(
            {
                "answer": answer,
                "answer_summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "turn_number": len(session.turns),
                "intent": parameters.get("intent"),
                "tool": tool.get("name"),
                "tool_status": tool.get("status"),
                "tool_summary": tool,
                "standalone": parameters.get("standalone"),
                "related_turn_ids": list(
                    parameters.get("related_turn_ids", [])
                ),
                "history_turn_ids": list(
                    parameters.get("history_turn_ids", [])
                ),
                "retrieval_mode": parameters.get("retrieval_mode"),
                "sources": _source_names(turn.evidence),
                "evidence_count": len(turn.evidence),
                "files_scanned": tool.get("files_scanned"),
                "files_with_mentions": tool.get(
                    "files_with_mentions"
                ),
                "listed": tool.get("listed"),
                "passages_read": tool.get("passages_read"),
                "planned_passages": tool.get("passages"),
                "generation_events": (
                    len(native.get("model_calls", []))
                    if native.get("trace_available") else None
                ),
                "tool_internal_attempts": (
                    len(attempts) if isinstance(attempts, list)
                    else None
                ),
                "invalid_citations": parameters.get("invalid_citations", []),
                "completion": parameters.get("completion"),
                "usage": parameters.get("usage", {}),
                "finish_reason": parameters.get("finish_reason"),
                "trace": native,
            }
        )
        _grade_record(record)
        context["case_turn_ids"][case_id] = len(session.turns)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


def probe_route(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    disallow=(),
):
    """Test routing against real conversation history without execution.

    Parameters
    ----------
    context : dict
        Active context with previously saved conversation turns.
    case_id : str
        Unique diagnostic case ID.
    question : str
        Question to route without saving or answering.
    intent : str or None, optional
        Intended router classification.
    standalone : bool or None, optional
        Expected independence from previous turns.
    related_to : tuple of str, optional
        Required previous case IDs.
    forbid_related : tuple of str, optional
        Previous case IDs that must not be linked.
    disallow : tuple of str, optional
        Intents that must not be selected for this request.

    Returns
    -------
    dict
        Routing-only decision and any diagnosed mismatches.

    Notes
    -----
    This uses KíLín's own ``route_question`` and the active session's
    private router accessor. It does not save an answer or execute a tool.

    """
    if intent is not None and intent not in literature_intents():
        raise ValueError(f"Unknown intent: {intent}")
    required = _expected_links(context, related_to)
    forbidden = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "route-only",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": required,
        "forbidden_related_ids": forbidden,
        "disallowed_intents": list(disallow),
        "previous": _previous_snapshot(context),
        "trace_path": str(context["path"].relative_to(project_dir)),
    }
    try:
        config = project_type(session.project.profile.type_id)
        mark = session.usage_mark()
        decision = route_question(
            session._router_chat(), config, session.turns, question
        )
        record.update(
            {
                "intent": decision.intent,
                "tool": None,
                "standalone": decision.standalone,
                "related_turn_ids": list(decision.related_turn_ids),
                "retrieval_mode": config.intents[
                    decision.intent
                ].retrieval_mode,
                "sources": [],
                "answer": "(routing only: no answer)",
                "usage": session.usage_since(mark),
                "trace": {
                    "shortcut": decision.shortcut,
                    "fallback": decision.fallback,
                    "router_raw": decision.raw,
                    "dropped_turns": list(decision.dropped_turns),
                    "refers_back": decision.refers_back,
                },
            }
        )
        _grade_record(record)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record

In [6]:
def show_results(*, prefix=None, group=None, mode=None):
    """Display a compact summary across selected recorded cases.

    Parameters
    ----------
    prefix : str or None, optional
        Limit case IDs to a prefix.
    group : str or None, optional
        Limit records to a named conversation group.
    mode : str or None, optional
        Limit to ``full`` or ``route-only`` cases.

    Returns
    -------
    list of dict
        Selected records.

    """
    rows = [
        record
        for record in CASE_RESULTS
        if (prefix is None or record["id"].startswith(prefix))
        and (group is None or record["label"] == group)
        and (mode is None or record["mode"] == mode)
    ]
    if not rows:
        print("No matching cases recorded.")
        return []
    headings = (
        "Case",
        "Model",
        "Expected",
        "Actual",
        "Links",
        "Route",
        "Scope",
        "Evidence files",
        "Warnings",
        "Seconds",
    )
    header = "".join(f"<th>{html.escape(name)}</th>" for name in headings)
    body = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 65)}</td>"
            for value in (
                item["id"],
                item.get("model", "-"),
                item.get("expected_intent") or "---",
                item.get("intent") or "-",
                str(item.get("related_turn_ids", [])),
                item["status"],
                item.get("scope_status", "---"),
                (
                    f"{len(item.get('sources', []))} retained / "
                    f"{item['files_scanned']} scanned"
                    if item.get("files_scanned") is not None
                    else str(len(item.get("sources", [])))
                    if item["mode"] == "full" else "---"
                ),
                ", ".join(
                    key for key in ("execution", "trace", "content")
                    if item.get("checks", {}).get(key)
                ) or "---",
                f"{item['seconds']:.2f}",
            )
        )
        + "</tr>"
        for item in rows
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody>"
            "</table></div>"
        )
    )
    return rows


def show_coverage():
    """Report coverage against all configured literature intents.

    Returns
    -------
    dict
        Intent-to-observation mapping for the active experiment.

    """
    coverage = {}
    for intent, settings in literature_intents().items():
        records = [
            item for item in CASE_RESULTS if item["expected_intent"] == intent
        ]
        statuses = [item["status"] for item in records]
        coverage[intent] = {
            "tool": settings.get("tool", {}).get("name"),
            "full": sum(item["mode"] == "full" for item in records),
            "probes": sum(item["mode"] == "route-only" for item in records),
            "matched": statuses.count("MATCH"),
            "review": statuses.count("REVIEW"),
            "errors": statuses.count("ERROR"),
        }
    heading = "".join(
        f"<th>{html.escape(name)}</th>"
        for name in (
            "Intent",
            "Expected tool",
            "Full",
            "Probes",
            "Match",
            "Review",
            "Errors",
        )
    )
    cells = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value)}</td>"
            for value in (
                intent,
                item["tool"] or "---",
                item["full"],
                item["probes"],
                item["matched"],
                item["review"],
                item["errors"],
            )
        )
        + "</tr>"
        for intent, item in coverage.items()
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{heading}</tr></thead><tbody>{cells}</tbody>"
            "</table></div>"
        )
    )
    return coverage


def show_answer_model_comparison():
    """Compare the same numbered prompts across answering models.

    Returns
    -------
    list of dict
        Matching full-pipeline answering-model records.

    """
    items = [item for item in CASE_RESULTS if item["id"].startswith("M_")]
    if not items:
        print("No answering-model comparison has been run.")
        return []
    items = sorted(
        items, key=lambda item: (item["id"].rsplit("_", 1)[-1], item["model"])
    )
    headers = (
        "Prompt",
        "Answer model",
        "Intent",
        "Sources",
        "Seconds",
        "Recorded summary",
        "Route",
    )
    head = "".join(f"<th>{html.escape(name)}</th>" for name in headers)
    rows = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 145)}</td>"
            for value in (
                item["id"].rsplit("_", 1)[-1],
                item["model"],
                item.get("intent"),
                ", ".join(item.get("sources", [])),
                f"{item['seconds']:.2f}",
                item.get("answer_summary") or "(summary unavailable)",
                item["status"],
            )
        )
        + "</tr>"
        for item in items
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:11px'>"
            "<table style='width:100%;border-collapse:collapse'>"
            f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody>"
            "</table></div>"
        )
    )
    return items


def show_diagnostic_totals():
    """Print routing outcomes and separate evidence/trace warnings.

    Returns
    -------
    dict
        Counts by execution mode and diagnostic category.

    """
    totals = {}
    for mode in ("full", "route-only"):
        items = [item for item in CASE_RESULTS if item["mode"] == mode]
        totals[mode] = {
            "cases": len(items),
            "match": sum(item["status"] == "MATCH" for item in items),
            "review": sum(item["status"] == "REVIEW" for item in items),
            "unscored": sum(item["status"] == "---" for item in items),
            "errors": sum(item["status"] == "ERROR" for item in items),
            "scope": sum(bool(item.get("checks", {}).get("scope"))
                         for item in items),
            "citation_warnings": sum(
                bool(item.get("invalid_citations")) for item in items
            ),
            "trace_warnings": sum(
                bool(item.get("checks", {}).get("trace"))
                for item in items
            ),
            "content_anchors": sum(
                bool(item.get("checks", {}).get("content"))
                for item in items
            ),
        }
    display(HTML(
        "<table style='font-size:12px;border-collapse:collapse'>"
        "<tr>" + "".join(f"<th>{name}</th>" for name in (
            "Mode", "Cases", "Match", "Review", "---", "Errors",
            "Scope flags", "Citation flags", "Trace flags",
            "Anchor flags"
        )) + "</tr>" + "".join(
            "<tr>" + "".join(f"<td>{_brief(value)}</td>" for value in (
                mode, item["cases"], item["match"], item["review"],
                item["unscored"], item["errors"], item["scope"],
                item["citation_warnings"], item["trace_warnings"],
                item["content_anchors"]
            )) + "</tr>" for mode, item in totals.items()
        ) + "</table>"
    ))
    return totals


def verify_saved_workspace():
    """Compare notebook full-turn records with committed trace checkpoints.

    Returns
    -------
    dict
        Number of verified answers and any field disagreements.

    Notes
    -----
    Routing probes have no saved turn and are deliberately excluded.

    """
    checked = 0
    problems = []
    cache = {}
    for record in CASE_RESULTS:
        if record["mode"] != "full" or record["status"] == "ERROR":
            continue
        path = project_dir / record["trace_path"]
        if path not in cache:
            try:
                with path.open(encoding="utf-8") as stream:
                    trace = json.load(stream)
                cache[path] = trace["sessions"]["05_conversations"][
                    "resume_state"
                ]["turns"]
            except (OSError, ValueError, KeyError, TypeError):
                cache[path] = []
        number = record.get("turn_number", 0)
        saved = (
            cache[path][number - 1]
            if 1 <= number <= len(cache[path]) else None
        )
        if saved is None:
            errors = ["missing committed turn"]
        else:
            params = saved.get("parameters", {})
            errors = []
            for field, observed in (
                ("answer", record.get("answer")),
                ("question", record.get("question")),
            ):
                if saved.get(field) != observed:
                    errors.append(field)
            for field in ("intent", "related_turn_ids", "history_turn_ids"):
                if params.get(field) != record.get(field):
                    errors.append(field)
            if len(saved.get("evidence", [])) != record.get(
                "evidence_count"
            ):
                errors.append("evidence count")
            if _source_names(saved.get("evidence", [])) != record.get(
                "sources"
            ):
                errors.append("evidence sources")
            if (params.get("tool") or {}).get("name") != record.get(
                "tool"
            ):
                errors.append("tool")
            if params.get("retrieval_mode") != record.get(
                "retrieval_mode"
            ):
                errors.append("retrieval mode")
            if (params.get("representation") or {}).get(
                "summary"
            ) != record.get("answer_summary"):
                errors.append("recorded summary")
        record["workspace_mismatches"] = errors
        checked += 1
        if errors:
            problems.append({"case": record["id"], "fields": errors})
    print("Committed turns verified:", checked)
    print("Notebook/workspace discrepancies:", len(problems))
    for item in problems:
        print(f"  {item['case']}: {', '.join(item['fields'])}")
    return {"checked": checked, "mismatches": problems}


def annotate_case(case_id, *, scientific=None, note=""):
    """Attach a human scientific-quality verdict without model grading.

    Parameters
    ----------
    case_id : str
        Existing case ID.
    scientific : str or None, optional
        Manual rating such as ``correct``, ``mixed``, or ``incorrect``.
    note : str, optional
        Explain which source evidence was checked.

    """
    CASE_INDEX[case_id]["manual_review"] = {
        "scientific": scientific,
        "note": note,
    }


SCIENTIFIC_REFERENCE_CASES = (
    (
        "D05",
        "The original BRAPH already has a modular, object-oriented design.",
        "mijalkov17braph.pdf: abstract and Fig. 1",
    ),
    (
        "P02",
        "BRAPH 2 limitations must not be conflated with the separate "
        "memory-capacity study's assumptions.",
        "tiunn26braph2.pdf; mijalkov25computational.pdf",
    ),
    (
        "I02",
        "DIVA analyzes Raman spectroscopy data from plant-stress studies.",
        "patil25deep.pdf: methods",
    ),
    (
        "I03",
        "Figure value 79.7 is an axis value, not a percent decrease.",
        "mijalkov25computational.pdf: Fig. 1G",
    ),
    (
        "X02",
        "Direct MATLAB/Python interoperability is described as future work.",
        "tiunn26braph2.pdf: future development",
    ),
    (
        "X03",
        "A five-paper comparison must address all five distinct studies.",
        "All five indexed PDFs",
    ),
    (
        "V01",
        "The 2017 BRAPH software was designed for modular extension.",
        "mijalkov17braph.pdf: abstract and Fig. 1",
    ),
    (
        "V04",
        "The BRAPH 2 metadata lists MATLAB and C++.",
        "tiunn26braph2.pdf: metadata C6",
    ),
    (
        "V05",
        "The value 79.7 in Fig. 1G is not a percentage reduction.",
        "mijalkov25computational.pdf: Fig. 1G",
    ),
)


def show_scientific_references():
    """Display source-grounded facts for independent manual assessment.

    Returns
    -------
    list of dict
        Selected case records that have been executed.

    Notes
    -----
    Evidence coverage and keyword anchors are diagnostic only. Scientific
    correctness requires checking the complete answer against the PDFs.

    """
    rows = []
    reviewed = []
    for case_id, fact, locator in SCIENTIFIC_REFERENCE_CASES:
        record = CASE_INDEX.get(case_id)
        if record is None:
            coverage = "not run"
            anchors = "---"
            judgment = "pending"
        else:
            reviewed.append(record)
            wanted = record.get("expected_sources", [])
            retained = set(record.get("sources", []))
            coverage = (
                f"{len(set(wanted) & retained)}/{len(wanted)}"
                if wanted else "---"
            )
            required = record.get("expected_answer_terms", [])
            anchors = (
                "review" if record.get("checks", {}).get("content")
                else "found" if required else "---"
            )
            judgment = (
                record.get("manual_review", {}).get("scientific")
                or "pending"
            )
        values = (case_id, fact, locator, coverage, anchors, judgment)
        rows.append(
            "<tr>" + "".join(
                f"<td>{_brief(value, 140)}</td>" for value in values
            ) + "</tr>"
        )
    headings = (
        "Case", "Reference fact", "Source locator", "Evidence",
        "Anchors", "Manual review",
    )
    header = "".join(
        f"<th>{html.escape(name)}</th>" for name in headings
    )
    display(HTML(
        "<div style='overflow-x:auto;font-size:12px'>"
        "<table style='border-collapse:collapse;width:100%'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    ))
    return reviewed


def export_evaluation():
    """Write local JSON/CSV records and the experiment manifest.

    Returns
    -------
    pathlib.Path
        Directory holding the saved evaluation files.

    """
    folder = project_dir / "outputs" / "notebook_evaluation"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "cases.json").write_text(
        json.dumps(CASE_RESULTS, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    with (folder / "cases.csv").open(
        "w", encoding="utf-8", newline=""
    ) as stream:
        columns = [
            "id",
            "mode",
            "label",
            "model",
            "router_model",
            "expected_intent",
            "intent",
            "expected_tool",
            "tool",
            "standalone",
            "related_turn_ids",
            "history_turn_ids",
            "retrieval_mode",
            "sources",
            "evidence_count",
            "files_scanned",
            "files_with_mentions",
            "passages_read",
            "generation_events",
            "tool_internal_attempts",
            "status",
            "issues",
            "seconds",
            "trace_path",
            "answer_summary",
            "summary_source",
            "summary_consistent",
            "scope_status",
            "checks",
            "invalid_citations",
            "completion",
            "workspace_mismatches",
            "manual_review",
        ]
        assert len(columns) == len(set(columns))
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for record in CASE_RESULTS:
            writer.writerow(
                {
                    name: (
                        json.dumps(record.get(name), ensure_ascii=False)
                        if isinstance(record.get(name), (list, dict))
                        else record.get(name)
                    )
                    for name in columns
                }
            )
    manifest = {
        "experiment_id": EXPERIMENT_ID,
        "project": str(project_dir),
        "primary_model": MODEL_IDS[PRIMARY_MODEL],
        "router_model": ROUTER_MODEL,
        "comparison_models": [MODEL_IDS[k] for k in COMPARISON_MODELS],
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
        "document_hashes": globals().get("FIXTURE_HASHES", {}),
        "literature_intents": list(literature_intents()),
        "summary_source": "conversation.turn.representation",
        "enabled_sections": {
            "long_tools": RUN_LONG_TOOLS,
            "source_checks": RUN_SOURCE_CHECKS,
            "reports": RUN_REPORTS,
            "answer_models": RUN_MODEL_COMPARISON,
            "router_models": RUN_ROUTER_COMPARISON,
        },
        "conversations": CONVERSATION_INDEX,
    }
    (folder / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print("Exported:", folder)
    return folder

### 2.1 Protocol preflight

Confirm that the installed literature configuration contains the expected
**21 intents**. If it changes, stop and update the coverage matrix instead
of silently omitting new behavior. Expected tools are loaded from the
current project-type definition, not repeated in the notebook.

In [7]:
PLANNED_INTENTS = {
    "small_talk",
    "question_answering",
    "compare_sources",
    "report_generation",
    "conversation_followup",
    "rephrase_previous",
    "project_inventory",
    "mention_listing",
    "refine_results",
    "source_context",
    "paper_overview",
    "suggest_reference",
    "topic_listing",
    "fact_finding",
    "document_reading",
    "mention_classification",
    "reference_check",
    "research_briefing",
    "popular_science_summary",
    "technical_summary",
    "experimental_protocol",
}
configured = set(literature_intents())
assert PLANNED_INTENTS == configured, (
    "Literature intents changed. Review additions/removals before testing: "
    f"missing={sorted(PLANNED_INTENTS - configured)}, "
    f"new={sorted(configured - PLANNED_INTENTS)}"
)
print("Configured literature intents:", len(configured))
print("Full routing checks require comparing intent, tool and context.")
show_coverage();

Configured literature intents: 21
Full routing checks require comparing intent, tool and context.


Intent,Expected tool,Full,Probes,Match,Review,Errors
question_answering,---,0,0,0,0,0
fact_finding,find_fact,0,0,0,0,0
compare_sources,---,0,0,0,0,0
document_reading,batch_review,0,0,0,0,0
paper_overview,paper_overview,0,0,0,0,0
reference_check,reference_check,0,0,0,0,0
suggest_reference,suggest_reference,0,0,0,0,0
research_briefing,generate_paper_summary,0,0,0,0,0
popular_science_summary,generate_paper_summary,0,0,0,0,0
technical_summary,generate_paper_summary,0,0,0,0,0


## 3. Create a fresh, initially empty literature workspace

This establishes a clean baseline. No source files are added yet. Starting
empty is important: it tests whether the assistant correctly distinguishes
conversation/general knowledge from evidence-backed literature claims.

The directory is protected against overwriting. Rerunning this creation cell
requires choosing a new experiment ID in Section 1.

In [8]:
from kilin.workspace import prepare_sources_workspace

if project_dir.exists():
    raise FileExistsError(
        f"Existing experiment: {project_dir}. "
        "Choose a new EXPERIMENT_ID; never overwrite prior results."
    )
project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
    settings={"allow_download": ALLOW_DOWNLOAD},
)
print("Created:", project.root)
print("Sources:", project.source)

Created: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-203250
Sources: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-203250/sources


### 3.1 Empty-project and unsupported-evidence tests

An empty workspace should never imply that project papers are available.
General-knowledge questions may be answered, but claims attributed to
project evidence must acknowledge the absence of indexed documents.
Inspect citation warnings and the complete responses independently of
routing success.

In [9]:
with conversation("empty-project") as ctx:
    run_case(
        ctx,
        "E01",
        "hello",
        intent="small_talk",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "E02",
        "What can you help me with?",
        intent=None,
        standalone=True,
    )
    run_case(
        ctx,
        "E03",
        "What is Brownian motion?",
        intent=None,
        standalone=True,
    )
    run_case(
        ctx,
        "E04",
        "Which articles are available in this project?",
        intent="project_inventory",
        retrieval="none",
    )
    run_case(
        ctx,
        "E05",
        "Summarize the articles in this project.",
        intent=None,
    )
    run_case(
        ctx,
        "E06",
        "Find articles about neural networks in this project.",
        intent="topic_listing",
    )

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Conversation empty-project: conversation_2026-11-02T19-32-51.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
small_talk → small_talk,--- → ---,need []; got [],none,0:,116/3,3.56s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
--- → small_talk,--- → ---,need []; got [],none,0:,2244/32,6.64s,---,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
--- → question_answering,--- → ---,need []; got [],search,0:,2308/1352,50.85s,---,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
project_inventory → project_inventory,project_inventory → p…,need []; got [],none,0:,2208/14,4.74s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
--- → project_inventory,--- → project_invento…,need []; got [],none,0:,2243/14,4.86s,---,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → topic_listing,find_passages → find_…,need []; got [],none,0:,2727/50,6.76s,MATCH,---


In [10]:
show_results(prefix="E");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
E01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,---,0,---,3.56
E02,Qwen/Qwen3-8B-MLX-4bit,---,small_talk,[],---,---,0,---,6.64
E03,Qwen/Qwen3-8B-MLX-4bit,---,question_answering,[],---,---,0,execution,50.85
E04,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,---,0,---,4.74
E05,Qwen/Qwen3-8B-MLX-4bit,---,project_inventory,[],---,---,0,---,4.86
E06,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,---,0,---,6.76


## 4. Add the five reference papers

This corpus contains the original BRAPH framework, BRAPH 2, a computational
memory-capacity study, a DIVA/Raman spectroscopy study, and an additional
network-science paper. Their different methods and data types are useful
for testing document identity, retrieval scope, and scientific comparisons.

The five files are deliberately fixed across experiments. Avoid changing
the corpus during a model comparison; otherwise retrieval differences
cannot be attributed to the model alone.

In [11]:
from hashlib import sha256

from kilin.workspace import add_project_sources

FIXTURE_NAMES = (
    "mijalkov17braph.pdf",
    "mijalkov25computational.pdf",
    "patil25deep.pdf",
    "tiunn26braph2.pdf",
    "yao25connecting.pdf",
)
fixture_dir = repo_root / "tests" / "fixtures"
articles = [fixture_dir / name for name in FIXTURE_NAMES]
missing = [p.name for p in articles if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Required test PDFs missing: {missing}")
project = add_project_sources(project_dir, articles)
FIXTURE_HASHES = {
    article.name: sha256(article.read_bytes()).hexdigest()
    for article in articles
}
for article in articles:
    print(
        f"{article.name:30s} SHA-256 prefix: "
        f"{FIXTURE_HASHES[article.name][:16]}"
    )
print("Imported PDFs:", len(list(project.source.glob("*.pdf"))))

mijalkov17braph.pdf            SHA-256 prefix: 8cf54f05b91d2cf9
mijalkov25computational.pdf    SHA-256 prefix: 09b7f5f693c45a36
patil25deep.pdf                SHA-256 prefix: 54295cf647568baf
tiunn26braph2.pdf              SHA-256 prefix: 17e83c8ded0f6707
yao25connecting.pdf            SHA-256 prefix: fe7ee76ca19acc9f
Imported PDFs: 5


## 5. Direct routing and document discovery

Start with standalone prompts whose targets are unambiguous. This section
compares file inventory, exact word mentions, paper overviews, semantic
retrieval, and comparison. In particular:

- **`mention_listing`** asks whether an exact word occurs.
- **`topic_listing`** identifies passages/files *by meaning* and can be much
  more expensive.
- **`paper_overview`** summarizes one paper;
- **`compare_sources`** compares sources;
- **`question_answering`** explains their scientific content.

We first test the inexpensive routes end-to-end, and reserve a full semantic
scan for the opt-in long-tool section.

In [12]:
with conversation("direct-routes") as ctx:
    run_case(
        ctx,
        "D01",
        "Which articles are available in this project?",
        intent="project_inventory",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "D02",
        "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
        intent="mention_listing",
        standalone=True,
        retrieval="none",
        choice="first20",
    )
    run_case(
        ctx,
        "D03",
        "Give me a short overview of tiunn26braph2.pdf.",
        intent="paper_overview",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "D04",
        "According to the BRAPH 2 paper, why was its software developed?",
        intent="question_answering",
        retrieval="search",
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "D05",
        "Compare the methods of mijalkov17braph.pdf "
        "and tiunn26braph2.pdf, including their differences.",
        intent="compare_sources",
        retrieval="search",
        sources=("mijalkov17braph.pdf", "tiunn26braph2.pdf"),
    )

Conversation direct-routes: conversation_2026-11-02T19-34-12.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
project_inventory → project_inventory,project_inventory → p…,need []; got [],none,0:,2124/14,5.61s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,1: tiunn26braph2.pdf (1 scanned),2437/35,5.69s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8559/653,55.83s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,--- → ---,need []; got [],search,"5: tiunn26braph2.pdf, yao25connecting.pdf, mij…",6018/533,37.17s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,--- → ---,"need []; got [3, 4]",search,"4: mijalkov17braph.pdf, tiunn26braph2.pdf, yao…",38347/901,301.38s,MATCH,---


In [13]:
show_results(prefix="D");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
D01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,---,0,---,5.61
D02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,1 retained / 1 scanned,---,5.69
D03,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,55.83
D04,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,OK,5,---,37.17
D05,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[3, 4]",MATCH,OK,4,---,301.38


## 6. Multi-turn conversational scope

These are **real saved-answer chains**, not independent classification prompts.
The same `RAGSession` is used throughout each chain. After every turn, the
notebook records its full answer, source filenames, one-based turn ID, and
router's selected related/history IDs.

A context test checks whether a follow-up refers to the *right previous turn*.
This is separate from whether the answer's content is scientifically correct.
For example, asking for DIVA methods must not silently change the active
paper back to the BRAPH memory-capacity study.

### 6.1 Ambiguous paper descriptions across a conversation

The phrase *deep-learning article* is ambiguous in a collection containing
several uses of neural networks. The sequence tests how the router resolves
that phrase and how later pronouns propagate the resulting interpretation.
This differs from Section 6.2, which uses explicit filenames.

Routing-only probes leave the conversation unchanged: every probe sees the
same preceding turns and recorded prior-answer summaries.

In [14]:
with conversation("paper-identity") as ctx:
    run_case(
        ctx,
        "P01",
        "Summarize the BRAPH 2 article.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "P02",
        "What are its main limitations?",
        intent="question_answering",
        standalone=False,
        related_to=("P01",),
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "P03",
        "How does it compare with the 2017 paper?",
        intent="compare_sources",
        standalone=False,
        related_to=("P01",),
    )
    run_case(
        ctx,
        "P04",
        "Now tell me about the deep learning article.",
        intent="paper_overview",
        standalone=False,
    )
    run_case(
        ctx,
        "P05",
        "What methods does it use?",
        intent="question_answering",
        standalone=False,
        related_to=("P04",),
        forbid_related=("P01",),
    )
    run_case(
        ctx,
        "P06",
        "Compare it with the previous article.",
        intent="compare_sources",
        standalone=False,
        related_to=("P04",),
    )
    probe_route(
        ctx,
        "P07",
        "What did the DIVA paper use as input?",
        intent="fact_finding",
        standalone=False,
        related_to=("P04",),
    )
    probe_route(
        ctx,
        "P08",
        "Which paper did we discuss first?",
        intent="conversation_followup",
        standalone=False,
    )

Conversation paper-identity: conversation_2026-11-02T19-41-59.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8068/744,58.35s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,--- → ---,need [1]; got [],search,"4: yao25connecting.pdf, tiunn26braph2.pdf, mij…",5996/594,39.64s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,--- → ---,"need [1]; got [1, 2]",search,"4: tiunn26braph2.pdf, mijalkov17braph.pdf, yao…",8210/983,66.09s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → question_answering,paper_overview → ---,need []; got [],search,"4: tiunn26braph2.pdf, patil25deep.pdf, mijalko…",6030/822,48.37s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,--- → ---,need [4]; got [],search,"4: mijalkov25computational.pdf, yao25connectin…",5890/1109,58.46s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,--- → ---,"need [4]; got [4, 5]",search,"4: tiunn26braph2.pdf, mijalkov25computational.…",7376/754,52.24s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
fact_finding → question_answering,---,need [4]; got [],search,0:,2589/16,5.56s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
conversation_followup → conversation_followup,---,need []; got [1],reuse,0:,2586/23,5.70s,MATCH,---


In [15]:
show_results(prefix="P");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
P01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,58.35
P02,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],REVIEW,OK,4,---,39.64
P03,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[1, 2]",MATCH,---,4,---,66.09
P04,Qwen/Qwen3-8B-MLX-4bit,paper_overview,question_answering,[],REVIEW,---,4,---,48.37
P05,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],REVIEW,---,4,---,58.46
P06,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[4, 5]",MATCH,---,4,---,52.24
P07,Qwen/Qwen3-8B-MLX-4bit,fact_finding,question_answering,[],REVIEW,---,---,---,5.56
P08,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[1],MATCH,---,---,---,5.70


### 6.2 Explicit document identity and controlled topic switches

An ambiguous description such as *the deep-learning paper* can refer to
several articles. Explicit filenames establish a clearer contract: a named
paper should remain the referent of subsequent pronouns until the user
changes topic. These tests exercise both a source switch and a return to an
earlier paper, independently of the paper's scientific correctness.

The required filenames are **retrieval-coverage checks**, while answer
keywords are minimal content anchors. Neither alone establishes that a claim
is supported by the paper.

In [16]:
with conversation("explicit-paper-identity") as ctx:
    run_case(
        ctx,
        "I01",
        "Summarize the DIVA Raman study in patil25deep.pdf.",
        intent="paper_overview",
        standalone=True,
        sources=("patil25deep.pdf",),
        answer_terms=("Raman",),
    )
    run_case(
        ctx,
        "I02",
        "What kind of measurements does that study analyze?",
        intent="question_answering",
        standalone=False,
        related_to=("I01",),
        sources=("patil25deep.pdf",),
        forbid_sources=("mijalkov25computational.pdf",),
        answer_terms=("Raman",),
    )
    run_case(
        ctx,
        "I03",
        "Now summarize mijalkov25computational.pdf instead.",
        intent="paper_overview",
        sources=("mijalkov25computational.pdf",),
        answer_terms=("memory",),
    )
    probe_route(
        ctx,
        "I04",
        "What architecture does this second study use?",
        intent="question_answering",
        standalone=False,
        related_to=("I03",),
        forbid_related=("I01",),
    )
    probe_route(
        ctx,
        "I05",
        "Returning to patil25deep.pdf, which inputs does DIVA use?",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "I06",
        "What methods does it use?",
        intent="question_answering",
        standalone=False,
        related_to=("I03",),
    )

Conversation explicit-paper-identity: conversation_2026-11-02T19-47-36.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: patil25deep.pdf,8229/723,57.63s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → fact_finding,--- → find_fact,need [1]; got [1],none,1: yao25connecting.pdf,3410/1287,49.76s,REVIEW,"scope, content"


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: mijalkov25computational.pdf,9446/624,58.45s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → fact_finding,---,need [3]; got [3],none,0:,2318/23,5.14s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
fact_finding → fact_finding,---,need []; got [1],none,0:,2326/23,5.12s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → fact_finding,---,need [3]; got [3],none,0:,2316/23,5.14s,REVIEW,---


In [17]:
show_results(prefix="I");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
I01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,OK,1,---,57.63
I02,Qwen/Qwen3-8B-MLX-4bit,question_answering,fact_finding,[1],REVIEW,REVIEW,1,content,49.76
I03,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,OK,1,---,58.45
I04,Qwen/Qwen3-8B-MLX-4bit,question_answering,fact_finding,[3],REVIEW,---,---,---,5.14
I05,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[1],MATCH,---,---,---,5.12
I06,Qwen/Qwen3-8B-MLX-4bit,question_answering,fact_finding,[3],REVIEW,---,---,---,5.14


### 6.3 Follow-up versus rephrase versus a fresh answer

A reference to *what the assistant just said* is different from a fresh
question about the article. The router exposes `standalone`, `refers_back`,
`related_turn_ids`, and `history_turn_ids`; examine them together.

In [18]:
with conversation("answer-continuity") as ctx:
    run_case(
        ctx,
        "C01",
        "Give me an overview of the BRAPH 2 paper.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "C02",
        "Summarize its main contributions and limitations.",
        intent="question_answering",
        standalone=False,
        related_to=("C01",),
    )
    run_case(
        ctx,
        "C03",
        "In your previous answer, what limitation did you mention first?",
        intent="conversation_followup",
        standalone=False,
        related_to=("C02",),
        retrieval="reuse",
    )
    run_case(
        ctx,
        "C04",
        "Rewrite that answer in two concise bullets "
        "without adding any new information.",
        intent="rephrase_previous",
        standalone=False,
        related_to=("C03",),
        retrieval="reuse",
    )
    probe_route(
        ctx,
        "C05",
        "How does the architecture work in BRAPH 2?",
        intent="question_answering",
    )
    probe_route(
        ctx,
        "C06",
        "Could you explain your second bullet?",
        intent="conversation_followup",
        standalone=False,
        related_to=("C04",),
    )

Conversation answer-continuity: conversation_2026-11-02T19-50-39.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8434/655,56.80s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → paper_overview,--- → paper_overview,need [1]; got [],none,1: tiunn26braph2.pdf,8574/812,61.69s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
conversation_followup → conversation_followup,--- → ---,need [2]; got [2],reuse,1: tiunn26braph2.pdf,9220/271,42.10s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
rephrase_previous → rephrase_previous,--- → ---,need [3]; got [2],reuse,1: tiunn26braph2.pdf,9067/1310,86.21s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,---,need []; got [],search,0:,2400/16,5.14s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
conversation_followup → conversation_followup,---,need [4]; got [4],reuse,0:,2396/23,5.26s,MATCH,---


In [19]:
show_results(prefix="C");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
C01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,56.80
C02,Qwen/Qwen3-8B-MLX-4bit,question_answering,paper_overview,[],REVIEW,---,1,---,61.69
C03,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[2],MATCH,---,1,---,42.10
C04,Qwen/Qwen3-8B-MLX-4bit,rephrase_previous,rephrase_previous,[2],REVIEW,---,1,---,86.21
C05,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,---,---,---,5.14
C06,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[4],MATCH,---,---,---,5.26


## 7. Previous-result tools and source boundaries

A request to filter **those results** should operate on an existing result
set; it should not silently start a new unrestricted corpus scan. These
cases distinguish three operations:

- `source_context` / `show_source`: inspect the context of a previously
  identified source passage;
- `topic_listing` / `find_passages`: find **new** passages by meaning;
- `refine_results` / `filter_results`: filter **previously listed results**.

Source-context tests include both an available prior citation and a request
that cannot be fulfilled merely by reusing unrelated history.

In [20]:
with conversation("result-scoping") as ctx:
    run_case(
        ctx,
        "S01",
        "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
        intent="mention_listing",
        standalone=True,
        choice="first20",
    )
    run_case(
        ctx,
        "S02",
        "Show the exact source text and surrounding "
        "context for the first cited sentence in that list.",
        intent="source_context",
        standalone=False,
        related_to=("S01",),
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "S03",
        "Of the BRAPH mentions you listed, "
        "which describe software features rather than "
        "bibliographic citations?",
        intent="refine_results",
        standalone=False,
        related_to=("S01",),
        forbid_sources=(
            "mijalkov17braph.pdf",
            "mijalkov25computational.pdf",
            "patil25deep.pdf",
            "yao25connecting.pdf",
        ),
    )
    probe_route(
        ctx,
        "S04",
        "Filter those earlier results to descriptions "
        "of graphical user interfaces.",
        intent="refine_results",
        standalone=False,
        related_to=("S03",),
    )
    probe_route(
        ctx,
        "S05",
        "Find new passages in tiunn26braph2.pdf "
        "explaining how its graphical interface works.",
        intent="topic_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "S06",
        "Show the source context for the first sentence you just quoted.",
        intent="source_context",
        standalone=False,
        related_to=("S02",),
    )
    probe_route(
        ctx,
        "S07",
        "Restrict the earlier twenty mentions to methods sections.",
        intent="refine_results",
        standalone=False,
        related_to=("S01",),
    )
    probe_route(
        ctx,
        "S08",
        "Find new passages discussing graphical interfaces, "
        "not just the earlier matches.",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "S09",
        "Read the sentence surrounding the first cited item.",
        intent="source_context",
        standalone=False,
        related_to=("S02",),
    )
    probe_route(
        ctx,
        "S10",
        "Search all five papers for exact BRAPH mentions.",
        intent="mention_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "S11",
        "Rewrite the filtered list as two concise paragraphs.",
        intent="rephrase_previous",
        standalone=False,
        related_to=("S03",),
    )

Conversation result-scoping: conversation_2026-11-02T19-54-58.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,1: tiunn26braph2.pdf (1 scanned),2369/35,6.74s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
source_context → source_context,show_source → show_so…,need [1]; got [1],none,1: tiunn26braph2.pdf,2577/478,21.34s,MATCH,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
refine_results → refine_results,filter_results → filt…,need [1]; got [1],none,1: tiunn26braph2.pdf,3348/64,10.21s,MATCH,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
refine_results → refine_results,---,"need [3]; got [1, 2, 3]",none,0:,2362/29,5.29s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → source_context,---,"need []; got [1, 2, 3]",none,0:,2370/28,5.33s,REVIEW,---


... (2 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → mention_listing,---,need []; got [1],none,0:,2364/22,5.18s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
source_context → source_context,---,need [2]; got [2],none,0:,2360/22,5.18s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,---,need []; got [1],none,0:,2361/22,5.19s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
rephrase_previous → refine_results,---,"need [3]; got [1, 2]",none,0:,2360/29,5.35s,REVIEW,---


In [21]:
show_results(prefix="S");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
S01,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,1 retained / 1 scanned,---,6.74
S02,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[1],MATCH,OK,1,execution,21.34
S03,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,[1],MATCH,OK,1,execution,10.21
S04,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,"[1, 2, 3]",MATCH,---,---,---,5.29
S05,Qwen/Qwen3-8B-MLX-4bit,topic_listing,source_context,"[1, 2, 3]",REVIEW,---,---,---,5.33
S06,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[2],MATCH,---,---,---,5.14
S07,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,"[1, 2, 3]",MATCH,---,---,---,5.29
S08,Qwen/Qwen3-8B-MLX-4bit,topic_listing,mention_listing,[1],REVIEW,---,---,---,5.18
S09,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[2],MATCH,---,---,---,5.18
S10,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[1],REVIEW,---,---,---,5.19


### 7.1 List-display, export, and noninteractive choices

Large result lists can be displayed, exported, or cancelled. These paths
exercise GUI-equivalent callbacks without a graphical frontend. The first
turn establishes a genuine antecedent for *these articles*. Inspect
`gui_events` to check the options and the chosen path. Artifact export is
opt-in to keep the default run lightweight.

In [22]:
with conversation("mention-choices") as ctx:
    run_case(
        ctx,
        "G00",
        "Which articles are available in this project?",
        intent="project_inventory",
    )
    run_case(
        ctx,
        "G01",
        "Which of these articles contain the literal term BRAPH?",
        intent="mention_listing",
        choice="first20",
    )
    run_case(
        ctx,
        "G02",
        "Which of these articles contain the literal term BRAPH?",
        intent="mention_listing",
        choice="none",
    )
    if RUN_GUI_EXPORT_TEST:
        run_case(
            ctx,
            "G03",
            "Which of these articles contain the literal term BRAPH?",
            intent="mention_listing",
            choice="save_all",
        )
    else:
        print("G03 omitted: RUN_GUI_EXPORT_TEST = False")

Conversation mention-choices: conversation_2026-11-02T19-56-20.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
project_inventory → project_inventory,project_inventory → p…,need []; got [],none,0:,2124/14,5.36s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,1: mijalkov17braph.pdf (5 scanned),2425/27,5.60s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,"4: mijalkov17braph.pdf, mijalkov25computationa…",2634/29,6.19s,MATCH,---


G03 omitted: RUN_GUI_EXPORT_TEST = False


In [23]:
show_results(prefix="G");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
G00,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,---,0,---,5.36
G01,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,1 retained / 5 scanned,---,5.60
G02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,4 retained / 5 scanned,---,6.19


## 8. Routing boundary probes (lightweight)

These probes test every configured literature intent without executing
specialized tools. They distinguish exact mentions from semantic search,
new evidence from inspection of earlier results, fact extraction from
literal keyword matching, and a claim check from a bibliography audit.

`---` in the expected-intent column means no single correct route is
specified. It is an exploratory observation, **not a pass**. Probes do not
add turns; tests in the same conversation therefore share the same saved
prior answers.

In [24]:
with conversation("routing-boundaries") as ctx:
    run_case(
        ctx,
        "B00",
        "Give me a short overview of tiunn26braph2.pdf.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "B00b",
        "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
        intent="mention_listing",
        choice="first20",
    )
    probe_route(
        ctx,
        "B01",
        "Does tiunn26braph2.pdf contain the exact word 'MATLAB'?",
        intent="mention_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "B02",
        "Find passages about the limitations of the "
        "architecture, even when the word limitations is absent.",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B03",
        "Find the passage in the BRAPH 2 paper "
        "describing its software architecture.",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B04",
        "Show me the context surrounding the first "
        "cited passage in your previous answer.",
        intent="source_context",
        standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx,
        "B05",
        "What programming language is BRAPH 2 implemented in?",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "B06",
        "Which papers use graph methods for "
        "brain-connectivity analysis, by meaning?",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B07",
        "Classify the five documents by their main research topic.",
        intent=None,
    )
    probe_route(
        ctx,
        "B08",
        "Group the previously listed sentences "
        "into software descriptions and citations.",
        intent="refine_results",
        standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx,
        "B08b",
        "Classify every mention of BRAPH in "
        "tiunn26braph2.pdf as a software description or citation.",
        intent="mention_classification",
        standalone=True,
    )
    probe_route(
        ctx,
        "B09",
        "Check whether the BRAPH 2 paper states that "
        "it includes a graphical user interface.",
        intent=None,
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "B10",
        "Audit the references cited by tiunn26braph2.pdf.",
        intent="reference_check",
    )
    probe_route(
        ctx,
        "B11",
        "Which paper should I cite for the "
        "introduction of the BRAPH framework?",
        intent="suggest_reference",
    )
    probe_route(
        ctx,
        "B12",
        "Read tiunn26braph2.pdf from start to "
        "finish and enumerate its components.",
        intent="document_reading",
    )
    probe_route(
        ctx,
        "B13",
        "Prepare a detailed technical summary of "
        "tiunn26braph2.pdf for specialists.",
        intent="technical_summary",
    )
    probe_route(
        ctx,
        "B13b",
        "Prepare a research briefing for tiunn26braph2.pdf.",
        intent="research_briefing",
    )
    probe_route(
        ctx,
        "B13c",
        "Write a popular science summary for "
        "tiunn26braph2.pdf for a general audience.",
        intent="popular_science_summary",
    )
    probe_route(
        ctx,
        "B13d",
        "Extract an experimental protocol from "
        "patil25deep.pdf, noting missing replication details.",
        intent="experimental_protocol",
    )
    probe_route(
        ctx,
        "B14",
        "How is BRAPH 2 different from the original BRAPH software?",
        intent="compare_sources",
    )
    probe_route(
        ctx,
        "B15",
        "Write a structured report comparing "
        "graph-analysis methods across these papers.",
        intent="report_generation",
    )
    probe_route(
        ctx,
        "B16",
        "List all programming languages used to implement "
        "tiunn26braph2.pdf.",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "B17",
        "Find the literal string MATLAB in tiunn26braph2.pdf.",
        intent="mention_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "B18",
        "Which earlier source introduced BRAPH originally?",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "B19",
        "Which source should I cite for the original BRAPH?",
        intent="suggest_reference",
    )
    probe_route(
        ctx,
        "B20",
        "Does the BRAPH 2 article cite the 2017 BRAPH paper?",
        intent="fact_finding",
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "B21",
        "Audit the bibliography of patil25deep.pdf and "
        "check whether its citations can be located.",
        intent="reference_check",
    )
    probe_route(
        ctx,
        "B22",
        "What did you say about BRAPH in your last answer?",
        intent="conversation_followup",
        standalone=False,
    )
    probe_route(
        ctx,
        "B23",
        "Rewrite your previous answer for a general audience.",
        intent="rephrase_previous",
        standalone=False,
    )
    probe_route(
        ctx,
        "B24",
        "Which articles in the project mention BRAPH2DiSTAP?",
        intent="mention_listing",
    )

Conversation routing-boundaries: conversation_2026-11-02T19-56-39.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8424/653,55.58s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,1: tiunn26braph2.pdf (1 scanned),2471/35,5.92s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,---,need []; got [2],none,0:,2303/22,5.04s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → topic_listing,---,need []; got [],none,0:,2302/25,5.09s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → source_context,---,need []; got [1],none,0:,2300/22,5.07s,REVIEW,---


... (21 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
reference_check → reference_check,---,need []; got [2],none,0:,2304/22,5.08s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
conversation_followup → conversation_followup,---,need []; got [2],reuse,0:,2297/23,5.13s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
rephrase_previous → popular_science_summary,---,need []; got [1],none,0:,2294/24,5.18s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
mention_listing → mention_listing,---,need []; got [2],none,0:,2298/22,5.14s,MATCH,---


In [25]:
show_results(prefix="B");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
B00,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,55.58
B00b,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,1 retained / 1 scanned,---,5.92
B01,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[2],REVIEW,---,---,---,5.04
B02,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,---,---,---,5.09
B03,Qwen/Qwen3-8B-MLX-4bit,topic_listing,source_context,[1],REVIEW,---,---,---,5.07
B04,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[2],MATCH,---,---,---,5.08
B05,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[],MATCH,---,---,---,4.93
B06,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,---,---,---,5.13
B07,Qwen/Qwen3-8B-MLX-4bit,---,report_generation,[],---,---,---,---,4.93
B08,Qwen/Qwen3-8B-MLX-4bit,refine_results,mention_classification,[2],REVIEW,---,---,---,5.07


## 9. Difficult conversational and source-scope cases

These scenarios distinguish a follow-up about an earlier answer from a new
source search, claim verification from a full bibliography audit, and a
single-paper question from a collection-wide comparison. Ambiguous requests
are retained as exploratory cases; explicit source names provide stricter
routing expectations. The expensive specialist tools remain opt-in.

In [26]:
with conversation("hard-scope") as ctx:
    run_case(
        ctx,
        "H01",
        "How does BRAPH 2 differ from the earlier BRAPH framework?",
        intent="compare_sources",
        standalone=True,
        sources=("mijalkov17braph.pdf", "tiunn26braph2.pdf"),
    )
    probe_route(
        ctx,
        "H02",
        "What evidence supports those differences?",
        intent="conversation_followup",
        standalone=False,
        related_to=("H01",),
    )
    probe_route(
        ctx,
        "H03",
        "Check whether the BRAPH 2 paper explicitly "
        "supports the claim that it provides a GUI.",
        intent=None,
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "H04",
        "List the datasets used by all five papers.",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "H05",
        "Do all five papers use the same dataset? Explain the differences.",
        intent="compare_sources",
    )
    probe_route(
        ctx,
        "H06",
        "Review the five articles for relevance to "
        "a literature review on computational brain connectivity.",
        intent=None,
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "H07",
        "Does tiunn26braph2.pdf use the same dataset as "
        "mijalkov25computational.pdf?",
        intent="compare_sources",
    )
    probe_route(
        ctx,
        "H08",
        "List the dataset of patil25deep.pdf specifically.",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "H09",
        "Compare the input data of the two BRAPH papers "
        "from the opening question.",
        intent="compare_sources",
        standalone=False,
    )
    probe_route(
        ctx,
        "H10",
        "Check the citation list of the BRAPH 2 paper.",
        intent="reference_check",
    )
    probe_route(
        ctx,
        "H11",
        "Find passages supporting the claim that "
        "BRAPH 2 provides a graphical interface.",
        intent="topic_listing",
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "H12",
        "List all five source filenames, not their topics.",
        intent="project_inventory",
    )

Conversation hard-scope: conversation_2026-11-02T20-00-05.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,--- → ---,need []; got [],search,"3: tiunn26braph2.pdf, mijalkov17braph.pdf, mij…",5421/786,45.35s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
conversation_followup → source_context,---,need [1]; got [1],none,0:,2206/22,4.83s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
--- → reference_check,---,need []; got [1],none,0:,2218/22,4.89s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
fact_finding → topic_listing,---,need []; got [],none,0:,2209/14,4.68s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,---,need []; got [1],search,0:,2213/22,4.88s,MATCH,---


... (3 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,---,need []; got [1],search,0:,2215/22,4.88s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
reference_check → reference_check,---,need []; got [1],none,0:,2212/22,4.89s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
topic_listing → source_context,---,need []; got [1],none,0:,2215/22,4.89s,REVIEW,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
project_inventory → project_inventory,---,need []; got [],none,0:,2210/14,4.73s,MATCH,---


In [27]:
show_results(prefix="H");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
H01,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,[],MATCH,OK,3,---,45.35
H02,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,source_context,[1],REVIEW,---,---,---,4.83
H03,Qwen/Qwen3-8B-MLX-4bit,---,reference_check,[1],REVIEW,---,---,---,4.89
H04,Qwen/Qwen3-8B-MLX-4bit,fact_finding,topic_listing,[],REVIEW,---,---,---,4.68
H05,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,[1],MATCH,---,---,---,4.88
H06,Qwen/Qwen3-8B-MLX-4bit,---,report_generation,[],---,---,---,---,4.71
H07,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,[1],MATCH,---,---,---,4.88
H08,Qwen/Qwen3-8B-MLX-4bit,fact_finding,mention_listing,[],REVIEW,---,---,---,4.72
H09,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,[1],MATCH,---,---,---,4.88
H10,Qwen/Qwen3-8B-MLX-4bit,reference_check,reference_check,[1],MATCH,---,---,---,4.89


### 9.1 Source-specific scientific spot checks

A correct intent does not guarantee that the answer uses the correct paper.
These tests check retrieval scope and a few important factual anchors:
**BRAPH2DiSTAP** is used in the Raman study, and BRAPH 2 presents direct
MATLAB/Python interoperability as future work. A comparison of all five
papers should retrieve evidence from all five, not merely a large number of
passages from a subset.

Source coverage and anchor checks help prioritize review; neither alone
establishes that every scientific statement is correct. Source-grounded
reference facts are listed in Section 13 for manual evaluation.

In [28]:
with conversation("scientific-scope") as ctx:
    run_case(
        ctx,
        "X01",
        "How is BRAPH2DiSTAP used in patil25deep.pdf, "
        "and what does the study measure?",
        intent="question_answering",
        retrieval="search",
        sources=("patil25deep.pdf",),
        forbid_sources=("mijalkov25computational.pdf",),
        answer_terms=("BRAPH2DiSTAP", "Raman"),
    )
    run_case(
        ctx,
        "X02",
        "Does tiunn26braph2.pdf report that direct "
        "Python integration is implemented or future work?",
        intent="question_answering",
        retrieval="search",
        sources=("tiunn26braph2.pdf",),
        answer_terms=("Python",),
    )
    run_case(
        ctx,
        "X03",
        "Compare the datasets used in all five "
        "project papers. Are they the same dataset?",
        intent="compare_sources",
        retrieval="search",
        sources=FIXTURE_NAMES,
    )

Conversation scientific-scope: conversation_2026-11-02T20-01-45.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,--- → ---,need []; got [],search,1: patil25deep.pdf,31725/570,212.85s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
question_answering → question_answering,--- → ---,need []; got [],search,1: tiunn26braph2.pdf,13828/456,75.60s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
compare_sources → compare_sources,--- → ---,"need []; got [1, 2]",search,"3: patil25deep.pdf, tiunn26braph2.pdf, mijalko…",43888/761,342.05s,MATCH,scope


In [29]:
show_results(prefix="X");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
X01,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,OK,1,---,212.85
X02,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,OK,1,---,75.60
X03,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[1, 2]",MATCH,REVIEW,3,---,342.05


### 9.2 Additional source-specific answer checks (opt-in)

These questions deliberately isolate individual papers before comparing
claims across documents. They are useful for evaluating whether retrieving a
correct source also produces correctly attributed conclusions. Keyword
anchors catch missing essential terms but **cannot certify entailment**;
read the complete answer and the PDF when interpreting the results.

The set includes a metadata-table question and a figure-reading check
to test whether answers preserve source-specific technical details.
Enable `RUN_SOURCE_CHECKS` in Section 1 to execute this group.

In [30]:
if RUN_SOURCE_CHECKS:
    with conversation("source-checks") as ctx:
        run_case(
            ctx,
            "V01",
            "Did the original BRAPH paper mijalkov17braph.pdf "
            "already support modular software extension?",
            intent="question_answering",
            sources=("mijalkov17braph.pdf",),
            answer_terms=("modular",),
        )
        run_case(
            ctx,
            "V02",
            "Which analytical examples are reported in "
            "tiunn26braph2.pdf? Name the example workflows.",
            intent="fact_finding",
            sources=("tiunn26braph2.pdf",),
        )
        run_case(
            ctx,
            "V03",
            "Which method uses Raman spectra in "
            "patil25deep.pdf, and for what purpose?",
            intent="question_answering",
            sources=("patil25deep.pdf",),
            answer_terms=("Raman", "DIVA"),
        )
        run_case(
            ctx,
            "V04",
            "Which programming languages does metadata table C6 "
            "of tiunn26braph2.pdf list?",
            intent="fact_finding",
            sources=("tiunn26braph2.pdf",),
            answer_terms=("MATLAB", "C++"),
            confirm="quick",
        )
        run_case(
            ctx,
            "V05",
            "In Figure 1G of mijalkov25computational.pdf, "
            "what does the value 79.7 represent?",
            intent="question_answering",
            sources=("mijalkov25computational.pdf",),
            answer_terms=("79.7",),
            confirm="quick",
        )
    show_results(prefix="V")
else:
    print("Additional source checks omitted.")

Additional source checks omitted.


## 10. Full end-to-end tool and intent coverage (opt-in)

Routing probes verify decisions but cannot establish that a specialized
tool executes successfully. This section runs the more expensive tools and
report templates if enabled in Section 1. Long-running tools may make many
internal model calls, especially with thinking enabled.

The scripted `quick` confirmation tests a bounded workflow, not exhaustive
processing. Network plans are always cancelled. Inspect partial results and
source quotations separately from the routing decision.

### 10.1 Extensive scans, classification, fact extraction, and references

Use this to test the real `find_passages`, `find_fact`, `batch_review`,
`batch_classify`, `suggest_reference`, and `reference_check` operations.
Check generated passages and quotation verification manually.

In [31]:
if RUN_LONG_TOOLS:
    with conversation("long-tools") as ctx:
        run_case(
            ctx,
            "L01",
            "Which papers apply graph-theory methods to "
            "analyze brain connectivity, based on meaning rather "
            "than keywords?",
            intent="topic_listing",
            confirm="quick",
        )
        run_case(
            ctx,
            "L02",
            "Give a complete list of the programming "
            "languages and software dependencies named in "
            "tiunn26braph2.pdf.",
            intent="fact_finding",
            confirm="quick",
        )
        run_case(
            ctx,
            "L03",
            "Read the whole tiunn26braph2.pdf paper "
            "from start to end and identify its components.",
            intent="document_reading",
            confirm="quick",
        )
        run_case(
            ctx,
            "L04",
            "Classify the mentions of BRAPH in "
            "tiunn26braph2.pdf into current software and citations.",
            intent="mention_classification",
            confirm="quick",
        )
        run_case(
            ctx,
            "L05",
            "Check the references cited by "
            "tiunn26braph2.pdf and determine which cited papers "
            "are present in this project.",
            intent="reference_check",
            confirm="quick",
        )
        run_case(
            ctx,
            "L06",
            "Which article should I cite for the "
            "introduction of the BRAPH framework?",
            intent="suggest_reference",
            confirm="quick",
        )
        run_case(
            ctx,
            "L07",
            "Suggest a reference from this project for "
            "the statement: BRAPH supports graph analysis of brain "
            "networks.",
            intent="suggest_reference",
            confirm="quick",
        )
        run_case(
            ctx,
            "L08",
            "Review the five articles for their "
            "relevance to computational brain connectivity.",
            intent="mention_classification",
            confirm="quick",
        )
else:
    print("Long-tool cases omitted: RUN_LONG_TOOLS = False")

Long-tool cases omitted: RUN_LONG_TOOLS = False


### 10.2 Named reporting templates

The named reporting intents share `generate_paper_summary` but select
different templates. A general comparative report is also tested. The
notebook records the selected intent, tool
summary (including template when present), saved output path, and native trace.
Review scientific specificity and fabricated method/benchmark claims in the
produced Markdown. No Qt PDF writer is supplied in notebook mode.

In [32]:
if RUN_REPORTS:
    with conversation("report-templates") as ctx:
        run_case(
            ctx,
            "T01",
            "Prepare a research briefing for tiunn26braph2.pdf.",
            intent="research_briefing",
            confirm="quick",
        )
        run_case(
            ctx,
            "T02",
            "Write a popular science summary of "
            "tiunn26braph2.pdf for a general audience.",
            intent="popular_science_summary",
            confirm="quick",
        )
        run_case(
            ctx,
            "T03",
            "Prepare a detailed technical summary of "
            "tiunn26braph2.pdf for specialists.",
            intent="technical_summary",
            confirm="quick",
        )
        run_case(
            ctx,
            "T04",
            "Extract an experimental protocol from "
            "tiunn26braph2.pdf; explicitly identify missing "
            "replication details instead of inventing them.",
            intent="experimental_protocol",
            confirm="quick",
        )
        run_case(
            ctx,
            "T05",
            "Write a structured report comparing the graph-analysis "
            "frameworks in mijalkov17braph.pdf and tiunn26braph2.pdf.",
            intent="report_generation",
            confirm="quick",
        )
else:
    print("Report templates omitted: RUN_REPORTS = False")

Report templates omitted: RUN_REPORTS = False


### 10.3 Explicit GUI cancellation and document selection

The tool callback should record **Cancel** on a confirmation plan without
proceeding to a long model run. U02 tests cancellation, not whether the
language list is scientifically complete; V04 tests that fact separately.
For ambiguous paper selection, the notebook
selects `tiunn26braph2.pdf` by its exact offered filename rather than always
taking the first option.

In [33]:
with conversation("gui-paths") as ctx:
    run_case(
        ctx,
        "U01",
        "Give me a short overview of one paper in this collection.",
        intent="paper_overview",
        choice=("document", "tiunn26braph2.pdf"),
    )
    run_case(
        ctx,
        "U02",
        "Give a complete list of the programming "
        "languages in tiunn26braph2.pdf.",
        intent="fact_finding",
        confirm="cancel",
    )

Conversation gui-paths: conversation_2026-11-02T20-12-18.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8777/629,59.40s,MATCH,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Other checks
fact_finding → mention_listing,find_fact → scan_and_…,need []; got [],none,0: (1 scanned),2470/35,5.82s,REVIEW,---


In [34]:
show_results(prefix="U");

Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
U01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,59.40
U02,Qwen/Qwen3-8B-MLX-4bit,fact_finding,mention_listing,[],REVIEW,---,0 retained / 1 scanned,---,5.82


## 11. Controlled answering-model comparison (opt-in)

The answering-model comparison holds the **five documents, router model,
questions, retrieval count, and project settings** fixed. Each answering model
gets a **fresh conversation**, preventing previous answers from one model
contaminating another. The three selected questions include an explicit
single-paper request, a two-paper comparison, and a context-dependent follow-up.

Only `PRIMARY_MODEL` and `COMPARISON_MODELS` are used. Models must already be
available locally when `ALLOW_DOWNLOAD=False`. Model switching is supported
for the MLX backend; changing from MLX to Torch requires a separate project
configuration, not simply `use_chat_model()`.

**Interpretation:** the router is fixed, but its related-turn decisions may
differ because each model's previous answers differ. Compare both the answer
and the recorded links, not just elapsed time.

In [35]:
if RUN_MODEL_COMPARISON:
    for model_key in dict.fromkeys(COMPARISON_MODELS):
        group = f"answer-model-{model_key}"
        with conversation(group, model=model_key) as ctx:
            run_case(
                ctx,
                f"M_{model_key}_01",
                "Give me an overview of the DIVA deep-learning "
                "paper patil25deep.pdf.",
                intent="paper_overview",
                standalone=True,
            )
            run_case(
                ctx,
                f"M_{model_key}_02",
                "What methods does it use, and what is the input "
                "data modality?",
                intent="question_answering",
                standalone=False,
                related_to=(f"M_{model_key}_01",),
            )
            run_case(
                ctx,
                f"M_{model_key}_03",
                "Compare those methods with the BRAPH 2 "
                "software-framework paper tiunn26braph2.pdf.",
                intent="compare_sources",
                standalone=False,
                related_to=(f"M_{model_key}_02",),
            )
    show_answer_model_comparison()
else:
    print("Answering-model comparison omitted.")

Answering-model comparison omitted.


## 12. Controlled router-model comparison (opt-in)

This comparison changes **only the router model**, with the same base
answering model and question wording. It uses *routing-only probes* to avoid
making the answer model perform expensive, irrelevant work.

One seed conversation is saved **once** under the default router. Its saved
history and source evidence are then **branched** into two independent
conversations using `ConversationStore.create(branch_from=...)`. Each router
therefore receives **the exact same prior answer and evidence**. Router-only
probes inspect the same questions against that identical history; they do not
add turns or run tools.

In [36]:
if RUN_ROUTER_COMPARISON:
    with conversation("router-seed") as seed_ctx:
        run_case(
            seed_ctx,
            "Rseed00",
            "Give me a short overview of tiunn26braph2.pdf.",
            intent="paper_overview",
        )
        parent_id = seed_ctx["info"].conversation_id

    for index, router_id in enumerate(COMPARISON_ROUTERS, start=1):
        group = f"router-model-{index}"
        with conversation(
            group, router=router_id, branch_from=parent_id
        ) as ctx:
            if len(ctx["session"].turns) != 1:
                raise RuntimeError("Router branch did not restore seed turn.")
            ctx["case_turn_ids"]["Rseed00"] = 1
            probe_route(
                ctx,
                f"R{index}_01",
                "Explain the second contribution you mentioned.",
                intent="conversation_followup",
                standalone=False,
                related_to=("Rseed00",),
            )
            probe_route(
                ctx,
                f"R{index}_02",
                "Which papers contain the exact word 'BRAPH'?",
                intent="mention_listing",
                standalone=True,
            )
            probe_route(
                ctx,
                f"R{index}_03",
                "Which papers discuss brain connectivity, by meaning?",
                intent="topic_listing",
            )
            probe_route(
                ctx,
                f"R{index}_04",
                "Check whether that paper supports the claim "
                "that it provides a GUI.",
                intent=None,
                standalone=False,
                related_to=("Rseed00",),
                disallow=("reference_check",),
            )
    show_results(prefix="R")
else:
    print("Router-model comparison omitted.")

Router-model comparison omitted.


## 13. Review and export the evaluation

Examine intent coverage and the separate routing, scope, citation, and
trace-diagnostic totals. Untested intents remain visible. A routing `MATCH`
means the declared routing contract passed; it says nothing by itself about
scientific correctness. Scope flags mean that expected retrieved evidence
was missing or that explicitly forbidden sources were present. Citation
warnings require inspection of source quotations.

The scientific-reference table gives specific facts and PDF locators
for manual adjudication, without turning keyword matches into a score.
Tool scan counts and retained evidence are distinct. Generation-stage
model events and tool-reported internal attempts are also separate.

Use `inspect_case("CASE_ID")` to inspect the complete answer and its
recorded summary, earlier saved representations, selected history, model
calls, and GUI decisions. Use `annotate_case()` to add a manual scientific
assessment after checking the source papers.

The final review also reconciles full answers, question text, selected
history IDs, and evidence counts against the committed workspace trace.
This detects notebook/export mismatches without rerunning inference.

In [37]:
reconciliation = verify_saved_workspace()
coverage = show_coverage()
diagnostics = show_diagnostic_totals()
all_end_to_end = show_results(mode="full")
all_routing = show_results(mode="route-only")
print("Total recorded cases:", len(CASE_RESULTS))
print("Unique conversations:", len(CONVERSATION_INDEX))

Committed turns verified: 38
Notebook/workspace discrepancies: 0


Intent,Expected tool,Full,Probes,Match,Review,Errors
question_answering,---,7,3,4,6,0
fact_finding,find_fact,1,8,2,7,0
compare_sources,---,5,4,9,0,0
document_reading,batch_review,0,1,1,0,0
paper_overview,paper_overview,8,0,7,1,0
reference_check,reference_check,0,3,3,0,0
suggest_reference,suggest_reference,0,2,2,0,0
research_briefing,generate_paper_summary,0,1,1,0,0
popular_science_summary,generate_paper_summary,0,1,1,0,0
technical_summary,generate_paper_summary,0,1,1,0,0


Mode,Cases,Match,Review,---,Errors,Scope flags,Citation flags,Trace flags,Anchor flags
full,38,28,7,3,0,2,3,0,1
route-only,54,31,20,3,0,0,0,0,0


Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
E01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,---,0,---,3.56
E02,Qwen/Qwen3-8B-MLX-4bit,---,small_talk,[],---,---,0,---,6.64
E03,Qwen/Qwen3-8B-MLX-4bit,---,question_answering,[],---,---,0,execution,50.85
E04,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,---,0,---,4.74
E05,Qwen/Qwen3-8B-MLX-4bit,---,project_inventory,[],---,---,0,---,4.86
E06,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,---,0,---,6.76
D01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,---,0,---,5.61
D02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,---,1 retained / 1 scanned,---,5.69
D03,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,---,1,---,55.83
D04,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,OK,5,---,37.17


Case,Model,Expected,Actual,Links,Route,Scope,Evidence files,Warnings,Seconds
P07,Qwen/Qwen3-8B-MLX-4bit,fact_finding,question_answering,[],REVIEW,---,---,---,5.56
P08,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[1],MATCH,---,---,---,5.70
I04,Qwen/Qwen3-8B-MLX-4bit,question_answering,fact_finding,[3],REVIEW,---,---,---,5.14
I05,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[1],MATCH,---,---,---,5.12
I06,Qwen/Qwen3-8B-MLX-4bit,question_answering,fact_finding,[3],REVIEW,---,---,---,5.14
C05,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,---,---,---,5.14
C06,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[4],MATCH,---,---,---,5.26
S04,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,"[1, 2, 3]",MATCH,---,---,---,5.29
S05,Qwen/Qwen3-8B-MLX-4bit,topic_listing,source_context,"[1, 2, 3]",REVIEW,---,---,---,5.33
S06,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[2],MATCH,---,---,---,5.14


Total recorded cases: 92
Unique conversations: 11


### 13.1 Source-grounded scientific checks

The table specifies reference facts and where they can be checked in
the source PDFs. **Evidence coverage and keyword anchors are not
scientific verdicts**: inspect the complete answers and record a manual
assessment using `annotate_case()`. Optional cases remain `not run`
until their sections are enabled. Routing results are not affected.


In [38]:
scientific_references = show_scientific_references()

Case,Reference fact,Source locator,Evidence,Anchors,Manual review
D05,"The original BRAPH already has a modular, object-oriented design.",mijalkov17braph.pdf: abstract and Fig. 1,2/2,---,pending
P02,BRAPH 2 limitations must not be conflated with the separate memory-capacity study's assumptions.,tiunn26braph2.pdf; mijalkov25computational.pdf,1/1,---,pending
I02,DIVA analyzes Raman spectroscopy data from plant-stress studies.,patil25deep.pdf: methods,0/1,review,pending
I03,"Figure value 79.7 is an axis value, not a percent decrease.",mijalkov25computational.pdf: Fig. 1G,1/1,found,pending
X02,Direct MATLAB/Python interoperability is described as future work.,tiunn26braph2.pdf: future development,1/1,found,pending
X03,A five-paper comparison must address all five distinct studies.,All five indexed PDFs,3/5,---,pending
V01,The 2017 BRAPH software was designed for modular extension.,mijalkov17braph.pdf: abstract and Fig. 1,not run,---,pending
V04,The BRAPH 2 metadata lists MATLAB and C++.,tiunn26braph2.pdf: metadata C6,not run,---,pending
V05,The value 79.7 in Fig. 1G is not a percentage reduction.,mijalkov25computational.pdf: Fig. 1G,not run,---,pending


### 13.2 Inspect failures and diagnostic warnings

Routing failures and execution errors are listed first. Source-coverage,
citation, trace, and content-anchor warnings are counted separately, so a
correct route is not penalized for a quotation that contains a bibliography
number. For manual assessment, use for example:

```python
annotate_case(
    "P05",
    scientific="incorrect",
    note="The answer attributes methods to the wrong article.",
)
```

A keyword anchor is only a sanity check. It does not replace reading the
source text or determining whether a cited passage supports a statement.

In [39]:
needs_review = [
    item for item in CASE_RESULTS if item["status"] in ("REVIEW", "ERROR")
]
for item in needs_review:
    print(
        f"{item['id']:<18} {item['status']:<7} "
        f"{', '.join(item.get('issues', []))}"
    )
print("Routing mismatches or execution errors:", len(needs_review))
other_flags = [
    item for item in CASE_RESULTS
    if any(item.get("checks", {}).get(name)
           for name in ("scope", "execution", "trace", "content"))
]
print("Separate scope/evidence/trace flags:", len(other_flags))

P02                REVIEW  routing: standalone, routing: missing prior-turn link
P04                REVIEW  routing: intent, routing: tool, routing: standalone
P05                REVIEW  routing: standalone, routing: missing prior-turn link
P07                REVIEW  routing: intent, routing: standalone, routing: missing prior-turn link
I02                REVIEW  routing: intent, routing: tool, scope: missing source in retrieved evidence, content: missing answer anchor: Raman
I04                REVIEW  routing: intent
I06                REVIEW  routing: intent
C02                REVIEW  routing: intent, routing: tool, routing: standalone, routing: missing prior-turn link
C04                REVIEW  routing: missing prior-turn link
S05                REVIEW  routing: intent, routing: standalone
S08                REVIEW  routing: intent
S10                REVIEW  routing: standalone
S11                REVIEW  routing: intent, routing: missing prior-turn link
B01                REVIEW  ro

### 13.3 Save the results

Exports remain under this experiment's `outputs/notebook_evaluation` folder:

- `cases.json`: complete records, answers, links, model settings, and flags;
- `cases.csv`: compact comparison table with distinct scan, evidence,
  and model-attempt fields;
- `manifest.json`: fixed configuration and conversation trace paths.

The original KíLín conversation JSON traces are preserved separately under
`logs/traces`. Keep the workspace when reporting regressions; a summary CSV
alone is insufficient to reproduce a routing or model failure.

In [40]:
evaluation_dir = export_evaluation()

Exported: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-203250/outputs/notebook_evaluation


## 14. Interpretation and next experiments

When a case fails, identify the responsible layer:

1. **Routing:** incorrect intent, standalone flag, or prior-turn links.
2. **Policy/tool:** correct route but unexpected retrieval policy or tool.
3. **Scope:** missing source PDF or unintended cross-document evidence.
4. **Execution/trace:** incomplete generation, token exhaustion, missing
   trace event, or trace/checkpoint disagreement.
5. **Scientific content:** correct routing and retrieved evidence, but an
   incorrect interpretation, missing comparison, or unsupported claim.

Use separate conversations and identical source hashes for model comparisons.
Make one change at a time, preserve native traces, and use source-grounded
regression cases before modifying KíLín's implementation.